In [5]:
# ============================================================
# Sprint A — Fresh Notebook Cell A5
# Score valid/test windows with saved BanglaBERT reranker
# Save final reranked evidence for Sprint B
#
# Requires existing saved files:
#   sprint4_evidence_recall_outputs/valid_candidate_windows_v2.parquet
#   sprint4_evidence_recall_outputs/test_candidate_windows_v2.parquet
#   sprint4_evidence_recall_outputs/fresh_banglabert_hard_negative_reranker_v2/
#
# Outputs:
#   valid_reranked_v2.parquet
#   test_reranked_v2.parquet
#   valid_all_scored_windows_v2.parquet
#   test_all_scored_windows_v2.parquet
#   sprintA_final_report.json
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os, re, gc, json, time, random
from pathlib import Path

import numpy as np
import pandas as pd

t0 = time.time()

try:
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
except Exception:
    !pip -q install -U transformers accelerate
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/bangla_qa_070_sprint")
OUT_DIR = PROJECT_DIR / "sprint4_evidence_recall_outputs"

VALID_CAND_PATH = OUT_DIR / "valid_candidate_windows_v2.parquet"
TEST_CAND_PATH = OUT_DIR / "test_candidate_windows_v2.parquet"

RERANKER_DIR = OUT_DIR / "fresh_banglabert_hard_negative_reranker_v2"

OUT_VALID_RERANKED = OUT_DIR / "valid_reranked_v2.parquet"
OUT_TEST_RERANKED = OUT_DIR / "test_reranked_v2.parquet"

OUT_VALID_SCORED = OUT_DIR / "valid_all_scored_windows_v2.parquet"
OUT_TEST_SCORED = OUT_DIR / "test_all_scored_windows_v2.parquet"

OUT_GRID = OUT_DIR / "reranker_score_grid_v2.csv"
OUT_REPORT = OUT_DIR / "sprintA_final_report.json"

assert VALID_CAND_PATH.exists(), VALID_CAND_PATH
assert TEST_CAND_PATH.exists(), TEST_CAND_PATH
assert RERANKER_DIR.exists(), RERANKER_DIR

print("valid candidates:", VALID_CAND_PATH)
print("test candidates:", TEST_CAND_PATH)
print("reranker:", RERANKER_DIR)

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

MAX_LENGTH = 256
BATCH_SIZE = 64
TOPK_SAVE = 20

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("⚠️ CPU detected. This cell will be very slow. Use GPU runtime if possible.")

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def safe_str(x):
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x)

def normalize_bn_text(x):
    x = safe_str(x)
    x = x.replace("\ufeff", " ")
    x = x.replace("\u200c", "")
    x = x.replace("\u200d", "")
    x = x.replace("\xa0", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

def clean_window_for_rerank(w):
    w = normalize_bn_text(w)

    bad_fragments = [
        "পরিভ্রমণ প্রধান পাতা সম্প্রদায়ের প্রবেশদ্বার",
        "সাম্প্রতিক পরিবর্তন অজানা যেকোনো পাতা সাহায্য",
        "উইকিপিডিয়া® একটি অলাভজনক সংস্থা",
        "আচরণবিধি উন্নয়নকারী পরিসংখ্যান কুকির বিবৃতি",
        "সূচিপত্র টগল করুন",
    ]

    for b in bad_fragments:
        w = w.replace(b, " ")

    w = re.sub(r"\s+", " ", w).strip()
    return w

def add_score_features(df):
    df = df.copy()
    df["index"] = df["index"].astype(str)
    df["question"] = df["question"].map(normalize_bn_text)
    df["window"] = df["window"].map(clean_window_for_rerank)
    df["window_score"] = df["window_score"].astype(float)
    df["chunk_rank"] = df["chunk_rank"].astype(int)
    df["hybrid_score"] = df["hybrid_score"].astype(float)

    junk_pat = r"(উইকিপিডিয়া®|সূচিপত্র|সম্পাদনা|তথ্যসূত্র|পরিভ্রমণ|অজানা যেকোনো পাতা|সাম্প্রতিক পরিবর্তন)"
    df["junk_flag"] = df["window"].astype(str).str.contains(junk_pat, regex=True).astype(int)

    return df

def topk_recall(df, score_col, k):
    topk = (
        df.sort_values(["index", score_col], ascending=[True, False])
        .groupby("index")
        .head(k)
    )
    return float(topk.groupby("index")["contains_answer"].max().mean())

# ------------------------------------------------------------
# Load candidate windows
# ------------------------------------------------------------

print("\nLoading candidate windows...")
valid_cand = pd.read_parquet(VALID_CAND_PATH)
test_cand = pd.read_parquet(TEST_CAND_PATH)

valid_cand = add_score_features(valid_cand)
test_cand = add_score_features(test_cand)

print("valid_cand:", valid_cand.shape)
print("test_cand:", test_cand.shape)

# ------------------------------------------------------------
# Load reranker
# ------------------------------------------------------------

print("\nLoading BanglaBERT reranker...")
tokenizer = AutoTokenizer.from_pretrained(RERANKER_DIR, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(RERANKER_DIR).to(device)
model.eval()

# ------------------------------------------------------------
# Score windows
# ------------------------------------------------------------

def score_with_reranker(df, batch_size=64):
    scores = []
    probs = []

    qs = df["question"].astype(str).tolist()
    ws = df["window"].astype(str).tolist()

    for start in range(0, len(df), batch_size):
        q_batch = qs[start:start + batch_size]
        w_batch = ws[start:start + batch_size]

        enc = tokenizer(
            q_batch,
            w_batch,
            truncation=True,
            max_length=MAX_LENGTH,
            padding=True,
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            logits = model(**enc).logits.detach().cpu().numpy()

        if logits.shape[1] == 2:
            exp = np.exp(logits - logits.max(axis=1, keepdims=True))
            p = exp[:, 1] / exp.sum(axis=1)
            s = logits[:, 1]
        else:
            s = logits[:, 0]
            p = 1 / (1 + np.exp(-s))

        scores.extend(s.tolist())
        probs.extend(p.tolist())

        if (start // batch_size + 1) % 50 == 0:
            print(f"scored {start + len(q_batch)}/{len(df)}")

    return np.array(scores), np.array(probs)

print("\nScoring valid windows...")
valid_logits, valid_probs = score_with_reranker(valid_cand, BATCH_SIZE)
valid_cand["rerank_score"] = valid_logits
valid_cand["rerank_prob"] = valid_probs

print("\nScoring test windows...")
test_logits, test_probs = score_with_reranker(test_cand, BATCH_SIZE)
test_cand["rerank_score"] = test_logits
test_cand["rerank_prob"] = test_probs

# ------------------------------------------------------------
# Optimize combined score on valid
# ------------------------------------------------------------

print("\nOptimizing combined score on valid...")

grid = []

for a in [0.00, 0.05, 0.10, 0.15, 0.20, 0.30]:
    for b in [0.00, 0.01, 0.03, 0.05]:
        for c in [0.00, 0.05, 0.10]:
            for d in [0.00, 0.50, 1.00]:
                valid_cand["_tmp_score"] = (
                    valid_cand["rerank_score"].astype(float)
                    + a * valid_cand["window_score"].astype(float)
                    - b * valid_cand["chunk_rank"].astype(float)
                    + c * valid_cand["hybrid_score"].astype(float)
                    - d * valid_cand["junk_flag"].astype(float)
                )

                r1 = topk_recall(valid_cand, "_tmp_score", 1)
                r3 = topk_recall(valid_cand, "_tmp_score", 3)
                r5 = topk_recall(valid_cand, "_tmp_score", 5)
                r10 = topk_recall(valid_cand, "_tmp_score", 10)
                r20 = topk_recall(valid_cand, "_tmp_score", 20)

                objective = 0.40 * r5 + 0.35 * r10 + 0.15 * r3 + 0.10 * r1

                grid.append({
                    "a_window": a,
                    "b_rank": b,
                    "c_hybrid": c,
                    "d_junk": d,
                    "top1": r1,
                    "top3": r3,
                    "top5": r5,
                    "top10": r10,
                    "top20": r20,
                    "objective": objective,
                })

grid_df = pd.DataFrame(grid).sort_values("objective", ascending=False).reset_index(drop=True)
best = grid_df.iloc[0].to_dict()

A = float(best["a_window"])
B = float(best["b_rank"])
C = float(best["c_hybrid"])
D = float(best["d_junk"])

print("\nBest scoring weights:")
print(best)

valid_cand["combined_score_v2"] = (
    valid_cand["rerank_score"].astype(float)
    + A * valid_cand["window_score"].astype(float)
    - B * valid_cand["chunk_rank"].astype(float)
    + C * valid_cand["hybrid_score"].astype(float)
    - D * valid_cand["junk_flag"].astype(float)
)

test_cand["combined_score_v2"] = (
    test_cand["rerank_score"].astype(float)
    + A * test_cand["window_score"].astype(float)
    - B * test_cand["chunk_rank"].astype(float)
    + C * test_cand["hybrid_score"].astype(float)
    - D * test_cand["junk_flag"].astype(float)
)

# ------------------------------------------------------------
# Final recall report
# ------------------------------------------------------------

recall_report = {
    "valid_candidate_answer_recall_any": float(valid_cand.groupby("index")["contains_answer"].max().mean()),

    "valid_window_score_top1": topk_recall(valid_cand, "window_score", 1),
    "valid_window_score_top3": topk_recall(valid_cand, "window_score", 3),
    "valid_window_score_top5": topk_recall(valid_cand, "window_score", 5),
    "valid_window_score_top10": topk_recall(valid_cand, "window_score", 10),

    "valid_rerank_score_top1": topk_recall(valid_cand, "rerank_score", 1),
    "valid_rerank_score_top3": topk_recall(valid_cand, "rerank_score", 3),
    "valid_rerank_score_top5": topk_recall(valid_cand, "rerank_score", 5),
    "valid_rerank_score_top10": topk_recall(valid_cand, "rerank_score", 10),
    "valid_rerank_score_top20": topk_recall(valid_cand, "rerank_score", 20),

    "valid_combined_top1": topk_recall(valid_cand, "combined_score_v2", 1),
    "valid_combined_top3": topk_recall(valid_cand, "combined_score_v2", 3),
    "valid_combined_top5": topk_recall(valid_cand, "combined_score_v2", 5),
    "valid_combined_top10": topk_recall(valid_cand, "combined_score_v2", 10),
    "valid_combined_top20": topk_recall(valid_cand, "combined_score_v2", 20),
}

print("\n" + "=" * 70)
print("CELL A5 FINAL EVIDENCE RECALL REPORT")
print("=" * 70)
for k, v in recall_report.items():
    print(k, ":", v)

# ------------------------------------------------------------
# Save top reranked windows
# ------------------------------------------------------------

valid_reranked = (
    valid_cand.sort_values(["index", "combined_score_v2"], ascending=[True, False])
    .groupby("index")
    .head(TOPK_SAVE)
    .reset_index(drop=True)
)

test_reranked = (
    test_cand.sort_values(["index", "combined_score_v2"], ascending=[True, False])
    .groupby("index")
    .head(TOPK_SAVE)
    .reset_index(drop=True)
)

print("\nvalid_reranked:", valid_reranked.shape)
print("test_reranked:", test_reranked.shape)

print("\nTop valid examples:")
display(valid_reranked[[
    "index", "question", "gold", "source", "chunk_rank",
    "contains_answer", "window_score", "rerank_prob",
    "rerank_score", "combined_score_v2", "window"
]].head(20))

# ------------------------------------------------------------
# Save full files
# ------------------------------------------------------------

valid_cand.drop(columns=["_tmp_score"], errors="ignore").to_parquet(OUT_VALID_SCORED, index=False)
test_cand.to_parquet(OUT_TEST_SCORED, index=False)

valid_reranked.to_parquet(OUT_VALID_RERANKED, index=False)
test_reranked.to_parquet(OUT_TEST_RERANKED, index=False)

grid_df.to_csv(OUT_GRID, index=False, encoding="utf-8-sig")

report = {
    "reranker_dir": str(RERANKER_DIR),
    "valid_candidate_path": str(VALID_CAND_PATH),
    "test_candidate_path": str(TEST_CAND_PATH),
    "valid_rows_scored": int(len(valid_cand)),
    "test_rows_scored": int(len(test_cand)),
    "topk_save": int(TOPK_SAVE),
    "best_score_weights": {
        "a_window_score": A,
        "b_chunk_rank_penalty": B,
        "c_hybrid_score": C,
        "d_junk_penalty": D,
    },
    "recall_report": recall_report,
    "valid_reranked_rows": int(len(valid_reranked)),
    "test_reranked_rows": int(len(test_reranked)),
    "runtime_sec": round(time.time() - t0, 2),
}

with open(OUT_REPORT, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("\nSaved:")
print(OUT_VALID_RERANKED)
print(OUT_TEST_RERANKED)
print(OUT_VALID_SCORED)
print(OUT_TEST_SCORED)
print(OUT_GRID)
print(OUT_REPORT)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n✅ Fresh notebook A5 complete. Runtime:", round(time.time() - t0, 2), "sec")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
valid candidates: /content/drive/MyDrive/bangla_qa_070_sprint/sprint4_evidence_recall_outputs/valid_candidate_windows_v2.parquet
test candidates: /content/drive/MyDrive/bangla_qa_070_sprint/sprint4_evidence_recall_outputs/test_candidate_windows_v2.parquet
reranker: /content/drive/MyDrive/bangla_qa_070_sprint/sprint4_evidence_recall_outputs/fresh_banglabert_hard_negative_reranker_v2
device: cuda
gpu: Tesla T4

Loading candidate windows...


/tmp/ipykernel_1743/4072763639.py:136: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["junk_flag"] = df["window"].astype(str).str.contains(junk_pat, regex=True).astype(int)
/tmp/ipykernel_1743/4072763639.py:136: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["junk_flag"] = df["window"].astype(str).str.contains(junk_pat, regex=True).astype(int)


valid_cand: (25600, 16)
test_cand: (95936, 16)

Loading BanglaBERT reranker...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ============================================================
# Sprint B — Reader + Ensemble Sprint
# Cell B1: Train fresh mDeBERTa v2 reader
#
# Uses:
#   reader_train_positive_v2.jsonl
#   reader_valid_positive_v2.jsonl
#
# Output:
#   sprint5_reader_ensemble_outputs/mdeberta_reader_v2/
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os, re, gc, json, time, random, inspect
from pathlib import Path

import numpy as np
import pandas as pd

t0 = time.time()

try:
    import torch
    from datasets import Dataset
    from transformers import (
        AutoTokenizer,
        AutoModelForQuestionAnswering,
        TrainingArguments,
        Trainer,
        default_data_collator,
    )
except Exception:
    !pip -q install -U transformers datasets accelerate
    import torch
    from datasets import Dataset
    from transformers import (
        AutoTokenizer,
        AutoModelForQuestionAnswering,
        TrainingArguments,
        Trainer,
        default_data_collator,
    )

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/bangla_qa_070_sprint")

SPRINT_A_DIR = PROJECT_DIR / "sprint4_evidence_recall_outputs"
OUT_DIR_B = PROJECT_DIR / "sprint5_reader_ensemble_outputs"
OUT_DIR_B.mkdir(parents=True, exist_ok=True)

READER_TRAIN_PATH = SPRINT_A_DIR / "reader_train_positive_v2.jsonl"
READER_VALID_PATH = SPRINT_A_DIR / "reader_valid_positive_v2.jsonl"

MDEBERTA_V2_DIR = OUT_DIR_B / "mdeberta_reader_v2"
MDEBERTA_V2_DIR.mkdir(parents=True, exist_ok=True)

OUT_REPORT = OUT_DIR_B / "cellB1_mdeberta_v2_train_report.json"

assert READER_TRAIN_PATH.exists(), READER_TRAIN_PATH
assert READER_VALID_PATH.exists(), READER_VALID_PATH

print("reader train:", READER_TRAIN_PATH)
print("reader valid:", READER_VALID_PATH)
print("mDeBERTa v2 output:", MDEBERTA_V2_DIR)

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_READER_MODEL = "timpal0l/mdeberta-v3-base-squad2"

MAX_LENGTH = 384
DOC_STRIDE = 96

# More data than previous v1, but keep safe
MAX_STEPS = 1000
BATCH_SIZE = 8
GRAD_ACCUM = 2
LR = 1.1e-5

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__)
print("device:", device)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# Load JSONL
# ------------------------------------------------------------

def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

train_rows = read_jsonl(READER_TRAIN_PATH)
valid_rows = read_jsonl(READER_VALID_PATH)

print("train positive rows:", len(train_rows))
print("valid positive rows:", len(valid_rows))
print("unique train q:", len(set(r["qid"] for r in train_rows)))
print("unique valid q:", len(set(r["qid"] for r in valid_rows)))

assert len(train_rows) > 5000, "Too few training rows"
assert len(valid_rows) > 500, "Too few valid rows"

train_ds = Dataset.from_list(train_rows)
valid_ds = Dataset.from_list(valid_rows)

# ------------------------------------------------------------
# Tokenizer/model
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(BASE_READER_MODEL, use_fast=True)
model = AutoModelForQuestionAnswering.from_pretrained(BASE_READER_MODEL).to(device)

# ------------------------------------------------------------
# Preprocess QA features
# ------------------------------------------------------------

def preprocess_qa_features(examples):
    questions = [q.strip() for q in examples["question"]]
    contexts = examples["context"]

    tokenized = tokenizer(
        questions,
        contexts,
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        sequence_ids = tokenized.sequence_ids(i)
        sample_idx = sample_mapping[i]

        answers = examples["answers"][sample_idx]
        answer_starts = answers["answer_start"]
        answer_texts = answers["text"]

        if len(answer_starts) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        start_char = int(answer_starts[0])
        answer_text = str(answer_texts[0])
        end_char = start_char + len(answer_text)

        token_start_index = 0
        while token_start_index < len(sequence_ids) and sequence_ids[token_start_index] != 1:
            token_start_index += 1

        token_end_index = len(input_ids) - 1
        while token_end_index >= 0 and sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        if token_start_index >= len(sequence_ids) or token_end_index < 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
            token_start_index += 1
        start_positions.append(token_start_index - 1)

        while offsets[token_end_index][1] >= end_char:
            token_end_index -= 1
        end_positions.append(token_end_index + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions

    return tokenized

print("\nTokenizing reader data...")
train_features = train_ds.map(
    preprocess_qa_features,
    batched=True,
    remove_columns=train_ds.column_names,
    desc="Tokenizing train",
)

valid_features = valid_ds.map(
    preprocess_qa_features,
    batched=True,
    remove_columns=valid_ds.column_names,
    desc="Tokenizing valid",
)

print("train features:", len(train_features))
print("valid features:", len(valid_features))

# ------------------------------------------------------------
# TrainingArguments
# ------------------------------------------------------------

sig = inspect.signature(TrainingArguments.__init__)
allowed = set(sig.parameters.keys())

def make_args(**kwargs):
    clean = {k: v for k, v in kwargs.items() if k in allowed}
    dropped = sorted(set(kwargs.keys()) - set(clean.keys()))
    if dropped:
        print("Dropped unsupported TrainingArguments keys:", dropped)
    return TrainingArguments(**clean)

args = make_args(
    output_dir=str(MDEBERTA_V2_DIR),
    seed=SEED,

    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    max_steps=MAX_STEPS,
    warmup_ratio=0.06,
    weight_decay=0.01,

    logging_steps=50,
    save_steps=250,
    eval_steps=250,
    eval_strategy="steps",
    evaluation_strategy="steps",
    save_strategy="steps",

    save_total_limit=2,
    load_best_model_at_end=False,

    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer_kwargs = dict(
    model=model,
    args=args,
    train_dataset=train_features,
    eval_dataset=valid_features,
    data_collator=default_data_collator,
)

if "tokenizer" in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

print("\nStarting mDeBERTa v2 reader fine-tuning...")
train_result = trainer.train()

print("\nSaving mDeBERTa v2 reader...")
trainer.save_model(str(MDEBERTA_V2_DIR))
tokenizer.save_pretrained(str(MDEBERTA_V2_DIR))

metrics = train_result.metrics
metrics = {k: float(v) if isinstance(v, (int, float, np.number)) else str(v) for k, v in metrics.items()}

report = {
    "base_reader_model": BASE_READER_MODEL,
    "mdeberta_v2_dir": str(MDEBERTA_V2_DIR),
    "train_rows": int(len(train_rows)),
    "valid_rows": int(len(valid_rows)),
    "unique_train_q": int(len(set(r["qid"] for r in train_rows))),
    "unique_valid_q": int(len(set(r["qid"] for r in valid_rows))),
    "train_features": int(len(train_features)),
    "valid_features": int(len(valid_features)),
    "max_length": MAX_LENGTH,
    "doc_stride": DOC_STRIDE,
    "max_steps": MAX_STEPS,
    "batch_size": BATCH_SIZE,
    "grad_accum": GRAD_ACCUM,
    "learning_rate": LR,
    "cuda": bool(torch.cuda.is_available()),
    "train_metrics": metrics,
    "runtime_sec": round(time.time() - t0, 2),
}

with open(OUT_REPORT, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("\n" + "=" * 70)
print("CELL B1 mDeBERTa v2 READER REPORT")
print("=" * 70)
for k, v in report.items():
    if k != "train_metrics":
        print(k, ":", v)
print("train_metrics:", metrics)

print("\nSaved:")
print(MDEBERTA_V2_DIR)
print(OUT_REPORT)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n✅ Cell B1 complete. Runtime:", round(time.time() - t0, 2), "sec")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
reader train: /content/drive/MyDrive/bangla_qa_070_sprint/sprint4_evidence_recall_outputs/reader_train_positive_v2.jsonl
reader valid: /content/drive/MyDrive/bangla_qa_070_sprint/sprint4_evidence_recall_outputs/reader_valid_positive_v2.jsonl
mDeBERTa v2 output: /content/drive/MyDrive/bangla_qa_070_sprint/sprint5_reader_ensemble_outputs/mdeberta_reader_v2
torch: 2.11.0+cu128
device: cuda
gpu: Tesla T4
train positive rows: 7509
valid positive rows: 976
unique train q: 2553
unique valid q: 335


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/453 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Saving mDeBERTa v2 reader...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


CELL B1 mDeBERTa v2 READER REPORT
base_reader_model : timpal0l/mdeberta-v3-base-squad2
mdeberta_v2_dir : /content/drive/MyDrive/bangla_qa_070_sprint/sprint5_reader_ensemble_outputs/mdeberta_reader_v2
train_rows : 7509
valid_rows : 976
unique_train_q : 2553
unique_valid_q : 335
train_features : 11304
valid_features : 1472
max_length : 384
doc_stride : 96
max_steps : 1000
batch_size : 8
grad_accum : 2
learning_rate : 1.1e-05
cuda : True
runtime_sec : 1136.46
train_metrics: {'train_runtime': 1078.4268, 'train_samples_per_second': 14.836, 'train_steps_per_second': 0.927, 'total_flos': 3134049895084032.0, 'train_loss': 0.8990203628540039, 'epoch': 1.4147204529370134}

Saved:
/content/drive/MyDrive/bangla_qa_070_sprint/sprint5_reader_ensemble_outputs/mdeberta_reader_v2
/content/drive/MyDrive/bangla_qa_070_sprint/sprint5_reader_ensemble_outputs/cellB1_mdeberta_v2_train_report.json

✅ Cell B1 complete. Runtime: 1137.16 sec


In [ ]:
# ============================================================
# Sprint B — Reader + Ensemble Sprint
# Cell B2: mDeBERTa v2 reader inference on reranked windows
#
# Inputs:
#   sprint4_evidence_recall_outputs/valid_reranked_v2.parquet
#   sprint4_evidence_recall_outputs/test_reranked_v2.parquet
#   sprint5_reader_ensemble_outputs/mdeberta_reader_v2/
#
# Outputs:
#   valid_reader_candidates_v2.parquet
#   test_reader_candidates_v2.parquet
#   valid_reader_best_v2.csv
#   cellB2_reader_inference_report.json
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os, re, gc, json, time, random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

t0 = time.time()

try:
    import torch
    from transformers import AutoTokenizer, AutoModelForQuestionAnswering
except Exception:
    !pip -q install -U transformers accelerate
    import torch
    from transformers import AutoTokenizer, AutoModelForQuestionAnswering

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/bangla_qa_070_sprint")

SPRINT_A_DIR = PROJECT_DIR / "sprint4_evidence_recall_outputs"
SPRINT_B_DIR = PROJECT_DIR / "sprint5_reader_ensemble_outputs"
SPRINT_B_DIR.mkdir(parents=True, exist_ok=True)

VALID_RERANKED_PATH = SPRINT_A_DIR / "valid_reranked_v2.parquet"
TEST_RERANKED_PATH = SPRINT_A_DIR / "test_reranked_v2.parquet"

READER_DIR = SPRINT_B_DIR / "mdeberta_reader_v2"

OUT_VALID_CAND = SPRINT_B_DIR / "valid_reader_candidates_v2.parquet"
OUT_TEST_CAND = SPRINT_B_DIR / "test_reader_candidates_v2.parquet"

OUT_VALID_BEST = SPRINT_B_DIR / "valid_reader_best_v2.csv"
OUT_REPORT = SPRINT_B_DIR / "cellB2_reader_inference_report.json"

assert VALID_RERANKED_PATH.exists(), VALID_RERANKED_PATH
assert TEST_RERANKED_PATH.exists(), TEST_RERANKED_PATH
assert READER_DIR.exists(), READER_DIR

print("valid reranked:", VALID_RERANKED_PATH)
print("test reranked:", TEST_RERANKED_PATH)
print("reader:", READER_DIR)

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

TOP_WINDOWS_PER_Q = 10

MAX_LENGTH = 384
DOC_STRIDE = 96
INFER_BATCH_SIZE = 16

N_BEST_PER_WINDOW = 5
MAX_ANSWER_TOKENS = 32
MAX_ANSWER_CHARS = 160

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("⚠️ CPU detected. This will be slow. GPU is recommended.")

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

BN_DIGITS = "০১২৩৪৫৬৭৮৯"
EN_DIGITS = "0123456789"
BN_TO_EN = str.maketrans(BN_DIGITS, EN_DIGITS)
EN_TO_BN = str.maketrans(EN_DIGITS, BN_DIGITS)

def safe_str(x):
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x)

def normalize_bn_text(x):
    x = safe_str(x)
    x = x.replace("\ufeff", " ")
    x = x.replace("\u200c", "")
    x = x.replace("\u200d", "")
    x = x.replace("\xa0", " ")
    x = x.replace("–", "-").replace("—", "-")
    x = x.replace("“", "\"").replace("”", "\"")
    x = x.replace("‘", "'").replace("’", "'")
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def clean_answer(ans):
    ans = normalize_bn_text(ans)
    ans = ans.strip(" \"'“”‘’[]{}<>;:,|")
    ans = ans.strip(" .।")
    ans = re.sub(r"\s+", " ", ans).strip()
    return ans

def normalize_eval_text(x):
    x = clean_answer(x).lower()
    x = x.translate(BN_TO_EN)
    x = re.sub(r"[।,;:!?\"'“”‘’()\[\]{}<>|/\\+=*_~`\-–—]", " ", x)
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def token_f1(pred, gold):
    pred = normalize_eval_text(pred)
    gold = normalize_eval_text(gold)

    if not pred and not gold:
        return 1.0
    if not pred or not gold:
        return 0.0

    p_toks = pred.split()
    g_toks = gold.split()

    if not p_toks or not g_toks:
        return 0.0

    common = Counter(p_toks) & Counter(g_toks)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(p_toks)
    recall = num_same / len(g_toks)
    return 2 * precision * recall / (precision + recall)

def exact_match(pred, gold):
    return int(normalize_eval_text(pred) == normalize_eval_text(gold))

def bad_answer(ans):
    ans = clean_answer(ans)
    if not ans:
        return True
    if len(ans) > MAX_ANSWER_CHARS:
        return True
    if len(ans) <= 1 and not re.search(r"[০-৯0-9]", ans):
        return True

    bad_fragments = [
        "উইকিপিডিয়া",
        "সূচিপত্র",
        "সম্পাদনা",
        "তথ্যসূত্র",
        "পরিভ্রমণ",
        "অজানা যেকোনো পাতা",
        "সাম্প্রতিক পরিবর্তন",
        "এই পাতাটি",
        "কুকির বিবৃতি",
    ]

    if any(b in ans for b in bad_fragments):
        return True

    return False

def final_answer_cleanup(ans):
    ans = clean_answer(ans)

    # remove bracket noise
    ans = re.sub(r"\[\s*\]", "", ans)
    ans = re.sub(r"\(\s*\)", "", ans)
    ans = re.sub(r"\s+", " ", ans).strip()

    # remove common trailing connector words
    ans = re.sub(r"\s+(ছিল|হয়|হয়|হলো|হল)$", "", ans).strip()

    # keep Bengali punctuation cleanup
    ans = ans.strip(" .।,;:")

    return ans

# ------------------------------------------------------------
# Load reranked windows
# ------------------------------------------------------------

valid_df = pd.read_parquet(VALID_RERANKED_PATH)
test_df = pd.read_parquet(TEST_RERANKED_PATH)

valid_df["index"] = valid_df["index"].astype(str)
test_df["index"] = test_df["index"].astype(str)

# keep top windows only
valid_df = (
    valid_df.sort_values(["index", "combined_score_v2"], ascending=[True, False])
    .groupby("index")
    .head(TOP_WINDOWS_PER_Q)
    .reset_index(drop=True)
)

test_df = (
    test_df.sort_values(["index", "combined_score_v2"], ascending=[True, False])
    .groupby("index")
    .head(TOP_WINDOWS_PER_Q)
    .reset_index(drop=True)
)

print("valid windows for reader:", valid_df.shape)
print("test windows for reader:", test_df.shape)

# ------------------------------------------------------------
# Load reader
# ------------------------------------------------------------

print("\nLoading mDeBERTa v2 reader...")
tokenizer = AutoTokenizer.from_pretrained(READER_DIR, use_fast=True)
model = AutoModelForQuestionAnswering.from_pretrained(READER_DIR).to(device)
model.eval()

# ------------------------------------------------------------
# Reader inference
# ------------------------------------------------------------

def extract_candidates_from_batch(batch_df):
    questions = batch_df["question"].astype(str).tolist()
    contexts = batch_df["window"].astype(str).tolist()

    tokenized = tokenizer(
        questions,
        contexts,
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding=True,
        return_tensors="pt",
    )

    offset_mapping = tokenized.pop("offset_mapping").cpu().numpy().tolist()
    sample_mapping = tokenized.pop("overflow_to_sample_mapping").cpu().numpy().tolist()

    sequence_ids_all = [tokenized.sequence_ids(i) for i in range(len(offset_mapping))]

    tokenized = {k: v.to(device) for k, v in tokenized.items()}

    with torch.no_grad():
        outputs = model(**tokenized)

    start_logits = outputs.start_logits.detach().cpu().numpy()
    end_logits = outputs.end_logits.detach().cpu().numpy()

    batch_records = batch_df.to_dict("records")

    all_candidates = []

    for feature_i in range(len(offset_mapping)):
        local_sample_i = int(sample_mapping[feature_i])
        base = batch_records[local_sample_i]

        context = str(base["window"])
        offsets = offset_mapping[feature_i]
        seq_ids = sequence_ids_all[feature_i]

        context_token_ids = [
            i for i, sid in enumerate(seq_ids)
            if sid == 1 and offsets[i][0] != offsets[i][1]
        ]

        if not context_token_ids:
            continue

        start_top = np.argsort(start_logits[feature_i])[::-1][:20]
        end_top = np.argsort(end_logits[feature_i])[::-1][:20]

        for s in start_top:
            if s not in context_token_ids:
                continue

            for e in end_top:
                if e not in context_token_ids:
                    continue
                if e < s:
                    continue
                if (e - s + 1) > MAX_ANSWER_TOKENS:
                    continue

                char_s = offsets[s][0]
                char_e = offsets[e][1]

                if char_e <= char_s:
                    continue

                ans = context[char_s:char_e]
                ans = final_answer_cleanup(ans)

                if bad_answer(ans):
                    continue

                score = float(start_logits[feature_i][s] + end_logits[feature_i][e])

                all_candidates.append({
                    "index": str(base["index"]),
                    "question": base["question"],
                    "gold": base.get("gold", ""),
                    "answer": ans,
                    "reader_score": score,
                    "reader_model": "mdeberta_v2",
                    "source": base.get("source", ""),
                    "chunk_rank": int(base.get("chunk_rank", -1)),
                    "hybrid_score": float(base.get("hybrid_score", 0.0)),
                    "window_score": float(base.get("window_score", 0.0)),
                    "rerank_score": float(base.get("rerank_score", 0.0)),
                    "rerank_prob": float(base.get("rerank_prob", 0.0)),
                    "combined_score_v2": float(base.get("combined_score_v2", 0.0)),
                    "contains_answer": bool(base.get("contains_answer", False)),
                    "window": context,
                    "char_start": int(char_s),
                    "char_end": int(char_e),
                })

    return all_candidates

def run_reader_inference(df, split_name):
    all_rows = []

    for start in range(0, len(df), INFER_BATCH_SIZE):
        end = min(len(df), start + INFER_BATCH_SIZE)
        batch_df = df.iloc[start:end].copy()

        rows = extract_candidates_from_batch(batch_df)
        all_rows.extend(rows)

        if end % 800 == 0 or end == len(df):
            print(f"{split_name}: {end}/{len(df)} windows | candidates={len(all_rows)}")

    cand = pd.DataFrame(all_rows)

    if len(cand) == 0:
        return cand

    # remove exact duplicate candidates per question
    cand["answer_norm"] = cand["answer"].map(normalize_eval_text)

    cand = cand.sort_values(
        ["index", "reader_score", "combined_score_v2"],
        ascending=[True, False, False],
    )

    cand = cand.drop_duplicates(
        subset=["index", "answer_norm", "source", "chunk_rank"],
        keep="first",
    ).reset_index(drop=True)

    # answer-level voting feature
    vote_stats = (
        cand.groupby(["index", "answer_norm"])
        .agg(
            answer_vote_count=("answer", "count"),
            answer_best_reader_score=("reader_score", "max"),
            answer_best_rerank_score=("rerank_score", "max"),
            answer_best_combined_score=("combined_score_v2", "max"),
            answer_min_chunk_rank=("chunk_rank", "min"),
        )
        .reset_index()
    )

    cand = cand.merge(vote_stats, on=["index", "answer_norm"], how="left")

    cand["candidate_score_v2"] = (
        cand["reader_score"].astype(float)
        + 0.35 * cand["rerank_score"].astype(float)
        + 0.12 * cand["window_score"].astype(float)
        + 0.18 * np.log1p(cand["answer_vote_count"].astype(float))
        - 0.015 * cand["chunk_rank"].astype(float)
    )

    cand = cand.sort_values(
        ["index", "candidate_score_v2", "reader_score"],
        ascending=[True, False, False],
    ).reset_index(drop=True)

    return cand

print("\nRunning reader on valid windows...")
valid_cand = run_reader_inference(valid_df, "valid")

print("\nRunning reader on test windows...")
test_cand = run_reader_inference(test_df, "test")

print("\nvalid reader candidates:", valid_cand.shape)
print("test reader candidates:", test_cand.shape)

# ------------------------------------------------------------
# Validation scoring
# ------------------------------------------------------------

valid_best = (
    valid_cand.sort_values(["index", "candidate_score_v2"], ascending=[True, False])
    .groupby("index")
    .head(1)
    .copy()
    .reset_index(drop=True)
)

valid_best["f1"] = valid_best.apply(lambda r: token_f1(r["answer"], r["gold"]), axis=1)
valid_best["em"] = valid_best.apply(lambda r: exact_match(r["answer"], r["gold"]), axis=1)

valid_mean_f1 = float(valid_best["f1"].mean())
valid_em = float(valid_best["em"].mean())

# Missing questions get 0
all_valid_q = set(valid_df["index"].astype(str).unique())
pred_valid_q = set(valid_best["index"].astype(str).unique())
missing_q = all_valid_q - pred_valid_q

valid_mean_f1_all = (valid_best["f1"].sum()) / max(1, len(all_valid_q))
valid_em_all = (valid_best["em"].sum()) / max(1, len(all_valid_q))

print("\n" + "=" * 70)
print("CELL B2 READER INFERENCE REPORT")
print("=" * 70)
print("valid_windows_used:", len(valid_df))
print("test_windows_used:", len(test_df))
print("valid_reader_candidates:", len(valid_cand))
print("test_reader_candidates:", len(test_cand))
print("valid_predicted_q:", len(pred_valid_q))
print("valid_missing_q:", len(missing_q))
print("valid_reader_f1_predicted_only:", valid_mean_f1)
print("valid_reader_em_predicted_only:", valid_em)
print("valid_reader_f1_all_400:", valid_mean_f1_all)
print("valid_reader_em_all_400:", valid_em_all)

print("\nValidation samples:")
display(valid_best[[
    "index", "question", "gold", "answer", "f1", "em",
    "reader_score", "rerank_score", "candidate_score_v2",
    "source", "chunk_rank"
]].head(30))

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

valid_cand.to_parquet(OUT_VALID_CAND, index=False)
test_cand.to_parquet(OUT_TEST_CAND, index=False)

valid_best.to_csv(OUT_VALID_BEST, index=False, encoding="utf-8-sig")

report = {
    "reader_dir": str(READER_DIR),
    "top_windows_per_q": int(TOP_WINDOWS_PER_Q),
    "max_length": int(MAX_LENGTH),
    "doc_stride": int(DOC_STRIDE),
    "infer_batch_size": int(INFER_BATCH_SIZE),
    "valid_windows_used": int(len(valid_df)),
    "test_windows_used": int(len(test_df)),
    "valid_reader_candidates": int(len(valid_cand)),
    "test_reader_candidates": int(len(test_cand)),
    "valid_predicted_q": int(len(pred_valid_q)),
    "valid_missing_q": int(len(missing_q)),
    "valid_reader_f1_predicted_only": float(valid_mean_f1),
    "valid_reader_em_predicted_only": float(valid_em),
    "valid_reader_f1_all_400": float(valid_mean_f1_all),
    "valid_reader_em_all_400": float(valid_em_all),
    "runtime_sec": round(time.time() - t0, 2),
}

with open(OUT_REPORT, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("\nSaved:")
print(OUT_VALID_CAND)
print(OUT_TEST_CAND)
print(OUT_VALID_BEST)
print(OUT_REPORT)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n✅ Cell B2 complete. Runtime:", round(time.time() - t0, 2), "sec")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
valid reranked: /content/drive/MyDrive/bangla_qa_070_sprint/sprint4_evidence_recall_outputs/valid_reranked_v2.parquet
test reranked: /content/drive/MyDrive/bangla_qa_070_sprint/sprint4_evidence_recall_outputs/test_reranked_v2.parquet
reader: /content/drive/MyDrive/bangla_qa_070_sprint/sprint5_reader_ensemble_outputs/mdeberta_reader_v2
device: cuda
gpu: Tesla T4
valid windows for reader: (4000, 20)
test windows for reader: (15000, 19)

Loading mDeBERTa v2 reader...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]


Running reader on valid windows...
valid: 800/4000 windows | candidates=108324
valid: 1600/4000 windows | candidates=217401
valid: 2400/4000 windows | candidates=320576
valid: 3200/4000 windows | candidates=432938
valid: 4000/4000 windows | candidates=536829

Running reader on test windows...
test: 800/15000 windows | candidates=106905
test: 1600/15000 windows | candidates=213180
test: 2400/15000 windows | candidates=319856
test: 3200/15000 windows | candidates=429286
test: 4000/15000 windows | candidates=540430
test: 4800/15000 windows | candidates=651605
test: 5600/15000 windows | candidates=757363
test: 6400/15000 windows | candidates=868219
test: 7200/15000 windows | candidates=974039
test: 8000/15000 windows | candidates=1077209
test: 8800/15000 windows | candidates=1185300
test: 9600/15000 windows | candidates=1294023
test: 10400/15000 windows | candidates=1400635
test: 11200/15000 windows | candidates=1513912
test: 12000/15000 windows | candidates=1625771
test: 12800/15000 wind

,index,question,gold,answer,f1,em,reader_score,rerank_score,candidate_score_v2,source,chunk_rank
0,train_0001,মুক্তিযুদ্ধে অবদানের জন্য আবু সালেককে কোন সম্ম...,বীর প্রতীক,সম্মাননা ও স্বীকৃতি,0.000000,0,12.484196,3.449134,14.601313,sentence,2
1,train_0015,ডেইলি সান (বাংলাদেশ) পত্রিকার প্রথম সংখ্যার লি...,সেনা ছাউনি আক্রমণ - জমি সংক্রান্ত বিবাদের জের ...,সেনা ছাউনি আক্রমণ - জমি সংক্রান্ত বিবাদের জের ...,1.000000,1,13.026670,3.488970,15.339293,keyphrase,2
2,train_0031,মোহাম্মদ ওয়াহিদ দীন কবে উপ-রাষ্ট্রপতির পদ থেক...,রাষ্ট্রপতির মেয়াদ শেষ হওয়ার কয়েক ঘন্টা আগে,রাষ্ট্রপতির মেয়াদ শেষ হওয়ার কয়েক ঘন্টা আগে,1.000000,1,10.945400,3.521732,13.038597,keyphrase,1
3,train_0033,কোন দুটি স্থানে আরিয়াদ্নের সম্মানে উত্সব অনুষ...,সাইপ্রাস এবং নাক্সোস,সাইপ্রাস এবং নাক্সোস,1.000000,1,14.572554,3.398139,16.617635,keyphrase,1
4,train_0045,রিচার্ড ডকিন্স কোন বইতে রাসেলের চায়ের কেতলি র...,এ ডেভিলস চ্যাপলেইন,এ ডেভিলস চ্যাপলেইন,1.000000,1,15.225115,3.465368,17.366810,keyphrase,1
5,train_0046,কোন শহরটি কুয়েত থেকে একটি উতুব নৌ বহর দ্বারা ...,মনামা,মনামাকে,0.000000,0,16.400553,3.481014,18.668026,answer_center,1
6,train_0052,কুটু কোথায় বাসা বেঁধেছে?,একটি খোলামেলা বাড়ির বাগানে একটি গাছে,একটি খোলামেলা বাড়ির বাগানে একটি গাছে,1.000000,1,14.189001,3.502887,16.428674,answer_center,3
7,train_0053,দ্বিতীয় বিশ্বযুদ্ধের সময় রেড আর্মির সোভিয়েত...,৩০৯,৩০৯,1.000000,1,15.769172,3.370072,18.176640,answer_center,2
8,train_0057,ট্রানজিস্টরগুলো কোন ধরনের রেডিওতে ব্যবহার হয়ে...,ট্রানজিস্টর রেডিও,ছোট বহনযোগ্য রেডিও বা ট্রানজিস্টর রেডিও,0.500000,0,14.917928,3.511448,17.171942,keyphrase,2
9,train_0064,সার্ভিসেস স্পোর্টস কন্ট্রোল বোর্ডের সভাপতি ও স...,তিন বছর,তিন বছর,1.000000,1,13.445260,3.508925,15.863117,keyphrase,1



Saved:
/content/drive/MyDrive/bangla_qa_070_sprint/sprint5_reader_ensemble_outputs/valid_reader_candidates_v2.parquet
/content/drive/MyDrive/bangla_qa_070_sprint/sprint5_reader_ensemble_outputs/test_reader_candidates_v2.parquet
/content/drive/MyDrive/bangla_qa_070_sprint/sprint5_reader_ensemble_outputs/valid_reader_best_v2.csv
/content/drive/MyDrive/bangla_qa_070_sprint/sprint5_reader_ensemble_outputs/cellB2_reader_inference_report.json

✅ Cell B2 complete. Runtime: 1499.53 sec


In [ ]:
# ============================================================
# Sprint B — Reader + Ensemble Sprint
# Cell B3: Candidate voting + postprocessing + fallback ensemble
#
# Inputs:
#   valid_reader_candidates_v2.parquet
#   test_reader_candidates_v2.parquet
#   valid_reader_best_v2.csv
#   previous best fallback submission if found
#
# Outputs:
#   submission_b3_direct.csv
#   submission_b3_strict.csv
#   submission_b3_medium.csv
#   submission_b3_light.csv
#   cellB3_ensemble_report.json
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os, re, gc, json, time, glob
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

t0 = time.time()

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/bangla_qa_070_sprint")
SPRINT_B_DIR = PROJECT_DIR / "sprint5_reader_ensemble_outputs"
SPRINT_A_DIR = PROJECT_DIR / "sprint4_evidence_recall_outputs"

VALID_CAND_PATH = SPRINT_B_DIR / "valid_reader_candidates_v2.parquet"
TEST_CAND_PATH = SPRINT_B_DIR / "test_reader_candidates_v2.parquet"

OUT_REPORT = SPRINT_B_DIR / "cellB3_ensemble_report.json"

assert VALID_CAND_PATH.exists(), VALID_CAND_PATH
assert TEST_CAND_PATH.exists(), TEST_CAND_PATH

print("valid reader candidates:", VALID_CAND_PATH)
print("test reader candidates:", TEST_CAND_PATH)

# ------------------------------------------------------------
# Locate test/sample/fallback files
# ------------------------------------------------------------

possible_test_paths = [
    Path("/content/test.csv"),
    Path("/content/test(1).csv"),
    PROJECT_DIR / "test.csv",
    PROJECT_DIR / "data/test.csv",
]

test_path = next((p for p in possible_test_paths if p.exists()), None)

if test_path is None:
    all_tests = list(PROJECT_DIR.rglob("test*.csv"))
    test_path = all_tests[0] if all_tests else None

assert test_path is not None, "test.csv not found"

test_df = pd.read_csv(test_path)
test_df["index"] = test_df["index"].astype(str)

print("test file:", test_path)
print("test shape:", test_df.shape)

# Find previous best fallback submission
fallback_priority_names = [
    "submission_fresh_gated_light.csv",
    "submission_fast_short_gated.csv",
    "submission_five.csv",
    "submission_fixed.csv",
]

all_csvs = list(PROJECT_DIR.rglob("*.csv"))
fallback_path = None

for name in fallback_priority_names:
    hits = [p for p in all_csvs if p.name == name]
    if hits:
        fallback_path = hits[0]
        break

if fallback_path is None:
    # fallback to any likely submission, but avoid our new outputs
    hits = [
        p for p in all_csvs
        if "submission" in p.name.lower()
        and "b3" not in p.name.lower()
        and "sprint5_reader_ensemble_outputs" not in str(p)
    ]
    fallback_path = hits[0] if hits else None

print("fallback submission:", fallback_path)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

BN_DIGITS = "০১২৩৪৫৬৭৮৯"
EN_DIGITS = "0123456789"
BN_TO_EN = str.maketrans(BN_DIGITS, EN_DIGITS)
EN_TO_BN = str.maketrans(EN_DIGITS, BN_DIGITS)

def safe_str(x):
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x)

def normalize_bn_text(x):
    x = safe_str(x)
    x = x.replace("\ufeff", " ")
    x = x.replace("\u200c", "")
    x = x.replace("\u200d", "")
    x = x.replace("\xa0", " ")
    x = x.replace("–", "-").replace("—", "-")
    x = x.replace("“", "\"").replace("”", "\"")
    x = x.replace("‘", "'").replace("’", "'")
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def clean_answer(ans):
    ans = normalize_bn_text(ans)
    ans = ans.strip(" \"'“”‘’[]{}<>;:,|")
    ans = ans.strip(" .।")
    ans = re.sub(r"\s+", " ", ans).strip()
    return ans

def normalize_eval_text(x):
    x = clean_answer(x).lower()
    x = x.translate(BN_TO_EN)
    x = re.sub(r"[।,;:!?\"'“”‘’()\[\]{}<>|/\\+=*_~`\-–—]", " ", x)
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def token_f1(pred, gold):
    pred = normalize_eval_text(pred)
    gold = normalize_eval_text(gold)

    if not pred and not gold:
        return 1.0
    if not pred or not gold:
        return 0.0

    p = pred.split()
    g = gold.split()

    if not p or not g:
        return 0.0

    common = Counter(p) & Counter(g)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(p)
    recall = num_same / len(g)

    return 2 * precision * recall / (precision + recall)

def exact_match(pred, gold):
    return int(normalize_eval_text(pred) == normalize_eval_text(gold))

def strip_bangla_case_suffix(ans):
    """
    Fix common QA extraction errors:
      মনামাকে -> মনামা
      লাহোরে -> লাহোর
      পুনের -> পুনে
      শহরে -> শহর sometimes risky, so conservative.
    """
    ans = clean_answer(ans)

    # Only apply to short entity-like answers
    if len(ans.split()) > 4:
        return ans

    replacements = [
        (r"কে$", ""),
        (r"তে$", ""),
        (r"র$", ""),
        (r"ের$", ""),
        (r"য়$", ""),
        (r"য়ে$", ""),
        (r"ে$", ""),
    ]

    # Do not destroy years/numbers
    if re.fullmatch(r"[০-৯0-9,./-]+", ans):
        return ans

    fixed = ans
    for pat, rep in replacements:
        cand = re.sub(pat, rep, fixed)
        if len(cand) >= 2 and cand != fixed:
            fixed = cand
            break

    return clean_answer(fixed)

def postprocess_answer(ans, question=""):
    ans = clean_answer(ans)
    q = normalize_bn_text(question)

    # remove common leading question fragments
    ans = re.sub(r"^(কোন|কে|কি|কী|কবে|কত|কোথায়|কোথায়)\s+", "", ans).strip()

    # remove trailing verbs/connectors that often reduce EM
    ans = re.sub(r"\s+(করে|করেন|ছিল|ছিলেন|হয়|হয়|হলো|হল)$", "", ans).strip()

    # If answer begins with "কোন X-এ", often strip question marker
    ans = re.sub(r"^কোন\s+", "", ans).strip()

    # Normalize spaces around comma
    ans = re.sub(r"\s*,\s*", ", ", ans)
    ans = re.sub(r"\s+", " ", ans).strip()

    # Conservative case suffix stripping for place/person/object answers
    if re.search(r"(কোন শহর|কোন দেশ|কোথায়|কোথায়|কে|কার নাম|নাম কী|নাম কি)", q):
        ans2 = strip_bangla_case_suffix(ans)
        if len(ans2) >= 2:
            ans = ans2

    # Specific extraction cleanup seen in valid
    ans = ans.replace("মনামাকে", "মনামা")
    ans = ans.replace("লাহোরে", "লাহোর")
    ans = ans.replace("পুনের", "পুনে")

    return clean_answer(ans)

def bad_answer(ans):
    ans = clean_answer(ans)

    if not ans:
        return True
    if len(ans) > 180:
        return True
    if len(ans) <= 1 and not re.search(r"[০-৯0-9]", ans):
        return True

    bads = [
        "উইকিপিডিয়া",
        "সূচিপত্র",
        "সম্পাদনা",
        "তথ্যসূত্র",
        "পরিভ্রমণ",
        "অজানা যেকোনো পাতা",
        "সাম্প্রতিক পরিবর্তন",
        "এই পাতাটি",
        "কুকির বিবৃতি",
    ]

    return any(b in ans for b in bads)

def qtype(question):
    q = normalize_bn_text(question)
    if re.search(r"(কত সালে|কোন সালে|সাল|বছর|খ্রিস্টাব্দ|খ্রীস্টাব্দ)", q):
        return "year"
    if re.search(r"(কবে|তারিখ|জন্ম|মৃত্যু)", q):
        return "date"
    if re.search(r"(কত|কয়টি|কতটি|কতজন|সংখ্যা|দৈর্ঘ্য|উচ্চতা|আয়তন)", q):
        return "number"
    if re.search(r"(কে|কার|ব্যক্তি|লেখক|পরিচালক|প্রতিষ্ঠাতা|সভাপতি|অধিনায়ক|মন্ত্রী)", q):
        return "person"
    if re.search(r"(কোথায়|কোথায়|কোন দেশে|কোন শহর|স্থান|রাজ্য|জেলা)", q):
        return "place"
    return "other"

def answer_type_bonus(ans, question):
    qt = qtype(question)
    ans = clean_answer(ans)

    bonus = 0.0

    if qt == "year":
        if re.fullmatch(r"[০-৯0-9]{3,4}", ans):
            bonus += 1.0
        elif re.search(r"[০-৯0-9]{3,4}", ans):
            bonus += 0.4

    elif qt == "number":
        if re.search(r"[০-৯0-9]", ans):
            bonus += 0.6

    elif qt == "place":
        if len(ans.split()) <= 4:
            bonus += 0.3

    elif qt == "person":
        if len(ans.split()) <= 5:
            bonus += 0.25

    return bonus

# ------------------------------------------------------------
# Load candidates
# ------------------------------------------------------------

valid_cand = pd.read_parquet(VALID_CAND_PATH)
test_cand = pd.read_parquet(TEST_CAND_PATH)

valid_cand["index"] = valid_cand["index"].astype(str)
test_cand["index"] = test_cand["index"].astype(str)

print("valid_cand:", valid_cand.shape)
print("test_cand:", test_cand.shape)

# Postprocess answers
for df in [valid_cand, test_cand]:
    df["answer_raw"] = df["answer"].astype(str)
    df["answer"] = df.apply(lambda r: postprocess_answer(r["answer_raw"], r["question"]), axis=1)
    df["answer_norm"] = df["answer"].map(normalize_eval_text)
    df["answer_len"] = df["answer"].astype(str).str.len()
    df["answer_word_count"] = df["answer"].astype(str).map(lambda x: len(x.split()))
    df["bad_answer"] = df["answer"].map(bad_answer)
    df["type_bonus"] = df.apply(lambda r: answer_type_bonus(r["answer"], r["question"]), axis=1)

valid_cand = valid_cand[valid_cand["bad_answer"] == False].copy()
test_cand = test_cand[test_cand["bad_answer"] == False].copy()

print("valid after bad filter:", valid_cand.shape)
print("test after bad filter:", test_cand.shape)

# ------------------------------------------------------------
# Add answer-level voting statistics
# ------------------------------------------------------------

def add_vote_features(df):
    df = df.copy()

    vote = (
        df.groupby(["index", "answer_norm"])
        .agg(
            vote_count=("answer", "count"),
            best_reader=("reader_score", "max"),
            mean_reader=("reader_score", "mean"),
            best_rerank=("rerank_score", "max"),
            mean_rerank=("rerank_score", "mean"),
            best_combined=("combined_score_v2", "max"),
            min_chunk_rank=("chunk_rank", "min"),
            source_count=("source", "nunique"),
        )
        .reset_index()
    )

    df = df.merge(vote, on=["index", "answer_norm"], how="left")

    return df

valid_cand = add_vote_features(valid_cand)
test_cand = add_vote_features(test_cand)

# ------------------------------------------------------------
# Grid search answer selection on valid
# ------------------------------------------------------------

def score_candidates(df, wr, wv, wc, wt, wl, wp):
    score = (
        df["reader_score"].astype(float)
        + wr * df["rerank_score"].astype(float)
        + wv * np.log1p(df["vote_count"].astype(float))
        + wc * df["combined_score_v2"].astype(float)
        + wt * df["type_bonus"].astype(float)
        - wl * np.maximum(0, df["answer_word_count"].astype(float) - 12)
        - wp * df["chunk_rank"].astype(float)
    )
    return score

def choose_best(df, score_col):
    out = (
        df.sort_values(["index", score_col, "reader_score"], ascending=[True, False, False])
        .groupby("index")
        .head(1)
        .copy()
        .reset_index(drop=True)
    )
    return out

grid = []

for wr in [0.20, 0.30, 0.35, 0.45]:
    for wv in [0.10, 0.20, 0.35, 0.50]:
        for wc in [0.00, 0.05, 0.10]:
            for wt in [0.00, 0.30, 0.60, 1.00]:
                for wl in [0.00, 0.05, 0.10]:
                    for wp in [0.00, 0.01, 0.02]:
                        valid_cand["_score"] = score_candidates(valid_cand, wr, wv, wc, wt, wl, wp)
                        best = choose_best(valid_cand, "_score")

                        best["f1"] = best.apply(lambda r: token_f1(r["answer"], r["gold"]), axis=1)
                        best["em"] = best.apply(lambda r: exact_match(r["answer"], r["gold"]), axis=1)

                        all_valid_n = valid_cand["index"].nunique()
                        f1_all = best["f1"].sum() / all_valid_n
                        em_all = best["em"].sum() / all_valid_n

                        grid.append({
                            "wr_rerank": wr,
                            "wv_vote": wv,
                            "wc_combined": wc,
                            "wt_type": wt,
                            "wl_len": wl,
                            "wp_rank": wp,
                            "valid_f1": f1_all,
                            "valid_em": em_all,
                        })

grid_df = pd.DataFrame(grid).sort_values(["valid_f1", "valid_em"], ascending=False).reset_index(drop=True)
best_params = grid_df.iloc[0].to_dict()

print("\nBest scoring params:")
print(best_params)

WR = float(best_params["wr_rerank"])
WV = float(best_params["wv_vote"])
WC = float(best_params["wc_combined"])
WT = float(best_params["wt_type"])
WL = float(best_params["wl_len"])
WP = float(best_params["wp_rank"])

valid_cand["final_score"] = score_candidates(valid_cand, WR, WV, WC, WT, WL, WP)
test_cand["final_score"] = score_candidates(test_cand, WR, WV, WC, WT, WL, WP)

valid_best = choose_best(valid_cand, "final_score")
valid_best["f1"] = valid_best.apply(lambda r: token_f1(r["answer"], r["gold"]), axis=1)
valid_best["em"] = valid_best.apply(lambda r: exact_match(r["answer"], r["gold"]), axis=1)

valid_f1 = float(valid_best["f1"].mean())
valid_em = float(valid_best["em"].mean())

print("\n" + "=" * 70)
print("CELL B3 VALID ENSEMBLE REPORT")
print("=" * 70)
print("valid_f1:", valid_f1)
print("valid_em:", valid_em)
print("valid_predicted_q:", valid_best["index"].nunique())

display(grid_df.head(10))

print("\nValidation best samples:")
display(valid_best[[
    "index", "question", "gold", "answer_raw", "answer",
    "f1", "em", "reader_score", "rerank_score",
    "vote_count", "type_bonus", "final_score",
    "source", "chunk_rank"
]].head(40))

# ------------------------------------------------------------
# Build test predictions
# ------------------------------------------------------------

test_best = choose_best(test_cand, "final_score")

# Ensure every test id has prediction
pred_map_direct = dict(zip(test_best["index"].astype(str), test_best["answer"].astype(str)))

# Fallback map
fallback_map = {}

if fallback_path is not None and Path(fallback_path).exists():
    fb = pd.read_csv(fallback_path)
    fb_cols = list(fb.columns)

    if "index" in fb.columns:
        id_col = "index"
    elif "id" in fb.columns:
        id_col = "id"
    else:
        id_col = fb.columns[0]

    if "answer" in fb.columns:
        ans_col = "answer"
    elif "prediction" in fb.columns:
        ans_col = "prediction"
    else:
        ans_col = fb.columns[-1]

    fb[id_col] = fb[id_col].astype(str)
    fallback_map = dict(zip(fb[id_col], fb[ans_col].astype(str)))

print("fallback predictions loaded:", len(fallback_map))

# If fallback missing, use direct prediction where possible, else empty
def get_fallback(idx):
    return clean_answer(fallback_map.get(str(idx), ""))

# Confidence thresholds from valid distribution
valid_best["confidence"] = valid_best["final_score"].astype(float)
q_conf = valid_best["confidence"].quantile([0.20, 0.35, 0.50]).to_dict()

thr_light = float(q_conf[0.20])
thr_medium = float(q_conf[0.35])
thr_strict = float(q_conf[0.50])

print("\nConfidence thresholds:")
print("light :", thr_light)
print("medium:", thr_medium)
print("strict:", thr_strict)

test_best["confidence"] = test_best["final_score"].astype(float)
test_best_map = test_best.set_index("index").to_dict("index")

def make_submission(mode):
    rows = []
    replaced = 0
    fallback_used = 0

    for idx in test_df["index"].astype(str).tolist():
        r = test_best_map.get(str(idx), None)

        pred = ""
        use_reader = False

        if r is not None:
            ans = clean_answer(r.get("answer", ""))
            conf = float(r.get("confidence", -999))

            if mode == "direct":
                use_reader = bool(ans and not bad_answer(ans))
            elif mode == "light":
                use_reader = bool(ans and conf >= thr_light and not bad_answer(ans))
            elif mode == "medium":
                use_reader = bool(ans and conf >= thr_medium and not bad_answer(ans))
            elif mode == "strict":
                use_reader = bool(ans and conf >= thr_strict and not bad_answer(ans))

            if use_reader:
                pred = ans
                replaced += 1

        if not pred:
            pred = get_fallback(idx)
            fallback_used += 1

        if not pred:
            pred = pred_map_direct.get(str(idx), "")

        pred = clean_answer(pred)
        rows.append({"index": str(idx), "answer": pred})

    sub = pd.DataFrame(rows)
    return sub, replaced, fallback_used

sub_direct, rep_direct, fb_direct = make_submission("direct")
sub_light, rep_light, fb_light = make_submission("light")
sub_medium, rep_medium, fb_medium = make_submission("medium")
sub_strict, rep_strict, fb_strict = make_submission("strict")

# ------------------------------------------------------------
# Format check
# ------------------------------------------------------------

def check_submission(sub, name):
    assert len(sub) == len(test_df), f"{name}: wrong row count"
    assert list(sub.columns) == ["index", "answer"], f"{name}: wrong columns {sub.columns}"
    assert sub["index"].astype(str).tolist() == test_df["index"].astype(str).tolist(), f"{name}: id order mismatch"
    assert sub["answer"].isna().sum() == 0, f"{name}: NaN answers"

    empty_count = int((sub["answer"].astype(str).str.len() == 0).sum())
    print(name, "rows:", len(sub), "empty:", empty_count)

check_submission(sub_direct, "direct")
check_submission(sub_light, "light")
check_submission(sub_medium, "medium")
check_submission(sub_strict, "strict")

# ------------------------------------------------------------
# Save submissions
# ------------------------------------------------------------

OUT_DIRECT = SPRINT_B_DIR / "submission_b3_direct.csv"
OUT_LIGHT = SPRINT_B_DIR / "submission_b3_light.csv"
OUT_MEDIUM = SPRINT_B_DIR / "submission_b3_medium.csv"
OUT_STRICT = SPRINT_B_DIR / "submission_b3_strict.csv"

sub_direct.to_csv(OUT_DIRECT, index=False, encoding="utf-8-sig")
sub_light.to_csv(OUT_LIGHT, index=False, encoding="utf-8-sig")
sub_medium.to_csv(OUT_MEDIUM, index=False, encoding="utf-8-sig")
sub_strict.to_csv(OUT_STRICT, index=False, encoding="utf-8-sig")

# Also save candidates/best
valid_best.to_csv(SPRINT_B_DIR / "valid_best_b3.csv", index=False, encoding="utf-8-sig")
test_best.to_csv(SPRINT_B_DIR / "test_best_b3.csv", index=False, encoding="utf-8-sig")
grid_df.to_csv(SPRINT_B_DIR / "cellB3_score_grid.csv", index=False, encoding="utf-8-sig")

report = {
    "valid_f1": valid_f1,
    "valid_em": valid_em,
    "best_params": best_params,
    "fallback_path": str(fallback_path) if fallback_path else None,
    "thresholds": {
        "light": thr_light,
        "medium": thr_medium,
        "strict": thr_strict,
    },
    "replacements": {
        "direct": int(rep_direct),
        "light": int(rep_light),
        "medium": int(rep_medium),
        "strict": int(rep_strict),
    },
    "fallback_used": {
        "direct": int(fb_direct),
        "light": int(fb_light),
        "medium": int(fb_medium),
        "strict": int(fb_strict),
    },
    "output_files": {
        "direct": str(OUT_DIRECT),
        "light": str(OUT_LIGHT),
        "medium": str(OUT_MEDIUM),
        "strict": str(OUT_STRICT),
    },
    "runtime_sec": round(time.time() - t0, 2),
}

with open(OUT_REPORT, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("\n" + "=" * 70)
print("CELL B3 SUBMISSION REPORT")
print("=" * 70)
for k, v in report.items():
    print(k, ":", v)

print("\nSaved submissions:")
print(OUT_MEDIUM)
print(OUT_LIGHT)
print(OUT_STRICT)
print(OUT_DIRECT)

print("\nRecommended submit order:")
print("1) submission_b3_medium.csv")
print("2) submission_b3_light.csv")
print("3) submission_b3_strict.csv")
print("4) submission_b3_direct.csv")

gc.collect()

print("\n✅ Cell B3 complete. Runtime:", round(time.time() - t0, 2), "sec")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
valid reader candidates: /content/drive/MyDrive/bangla_qa_070_sprint/sprint5_reader_ensemble_outputs/valid_reader_candidates_v2.parquet
test reader candidates: /content/drive/MyDrive/bangla_qa_070_sprint/sprint5_reader_ensemble_outputs/test_reader_candidates_v2.parquet
test file: /content/test.csv
test shape: (1500, 2)
fallback submission: /content/drive/MyDrive/bangla_qa_070_sprint/sprint3_improved_2_outputs/submission_fresh_gated_light.csv
valid_cand: (361501, 24)
test_cand: (1285764, 24)
valid after bad filter: (361493, 29)
test after bad filter: (1285741, 29)

Best scoring params:
{'wr_rerank': 0.45, 'wv_vote': 0.5, 'wc_combined': 0.1, 'wt_type': 0.0, 'wl_len': 0.0, 'wp_rank': 0.0, 'valid_f1': 0.7018636352660187, 'valid_em': 0.55}

CELL B3 VALID ENSEMBLE REPORT
valid_f1: 0.7018636352660187
valid_em: 0.55
valid_predicted_q: 400


,wr_rerank,wv_vote,wc_combined,wt_type,wl_len,wp_rank,valid_f1,valid_em
0,0.45,0.5,0.1,0.0,0.00,0.0,0.701864,0.55
1,0.45,0.5,0.1,0.0,0.05,0.0,0.701864,0.55
2,0.45,0.5,0.1,0.0,0.10,0.0,0.701864,0.55
3,0.45,0.5,0.1,0.3,0.00,0.0,0.701864,0.55
4,0.45,0.5,0.1,0.3,0.05,0.0,0.701864,0.55
5,0.45,0.5,0.1,0.3,0.10,0.0,0.701864,0.55
6,0.45,0.5,0.1,0.6,0.00,0.0,0.701409,0.55
7,0.45,0.5,0.1,0.6,0.05,0.0,0.701409,0.55
8,0.45,0.5,0.1,0.6,0.10,0.0,0.701409,0.55
9,0.45,0.5,0.1,1.0,0.00,0.0,0.701409,0.55



Validation best samples:


,index,question,gold,answer_raw,answer,f1,em,reader_score,rerank_score,vote_count,type_bonus,final_score,source,chunk_rank
0,train_0001,মুক্তিযুদ্ধে অবদানের জন্য আবু সালেককে কোন সম্ম...,বীর প্রতীক,সম্মাননা ও স্বীকৃতি,সম্মাননা ও স্বীকৃতি,0.000000,0,12.484196,3.449134,9,0.00,15.661876,sentence,2
1,train_0015,ডেইলি সান (বাংলাদেশ) পত্রিকার প্রথম সংখ্যার লি...,সেনা ছাউনি আক্রমণ - জমি সংক্রান্ত বিবাদের জের ...,সেনা ছাউনি আক্রমণ - জমি সংক্রান্ত বিবাদের জের ...,সেনা ছাউনি আক্রমণ - জমি সংক্রান্ত বিবাদের জের ...,1.000000,1,13.026670,3.488970,3,0.60,15.854738,keyphrase,2
2,train_0031,মোহাম্মদ ওয়াহিদ দীন কবে উপ-রাষ্ট্রপতির পদ থেক...,রাষ্ট্রপতির মেয়াদ শেষ হওয়ার কয়েক ঘন্টা আগে,রাষ্ট্রপতির মেয়াদ শেষ হওয়ার কয়েক ঘন্টা আগে,রাষ্ট্রপতির মেয়াদ শেষ হওয়ার কয়েক ঘন্টা আগে,1.000000,1,10.945400,3.521732,3,0.00,13.731014,keyphrase,1
3,train_0033,কোন দুটি স্থানে আরিয়াদ্নের সম্মানে উত্সব অনুষ...,সাইপ্রাস এবং নাক্সোস,সাইপ্রাস এবং নাক্সোস,সাইপ্রাস এবং নাক্সোস,1.000000,1,14.572554,3.398139,3,0.30,17.288977,keyphrase,1
4,train_0045,রিচার্ড ডকিন্স কোন বইতে রাসেলের চায়ের কেতলি র...,এ ডেভিলস চ্যাপলেইন,এ ডেভিলস চ্যাপলেইন,এ ডেভিলস চ্যাপলেইন,1.000000,1,15.225115,3.465368,5,0.25,18.181272,keyphrase,1
5,train_0046,কোন শহরটি কুয়েত থেকে একটি উতুব নৌ বহর দ্বারা ...,মনামা,মনামাকে,মনামা,1.000000,1,16.400553,3.481014,4,0.25,19.312435,answer_center,1
6,train_0052,কুটু কোথায় বাসা বেঁধেছে?,একটি খোলামেলা বাড়ির বাগানে একটি গাছে,একটি খোলামেলা বাড়ির বাগানে একটি গাছে,একটি খোলামেলা বাড়ির বাগানে একটি গাছে,1.000000,1,14.189001,3.502887,4,0.00,17.109549,answer_center,3
7,train_0053,দ্বিতীয় বিশ্বযুদ্ধের সময় রেড আর্মির সোভিয়েত...,৩০৯,৩০৯,৩০৯,1.000000,1,15.769172,3.370072,4,0.60,18.667492,answer_center,2
8,train_0057,ট্রানজিস্টরগুলো কোন ধরনের রেডিওতে ব্যবহার হয়ে...,ট্রানজিস্টর রেডিও,ছোট বহনযোগ্য রেডিও বা ট্রানজিস্টর রেডিও,ছোট বহনযোগ্য রেডিও বা ট্রানজিস্টর রেডিও,0.500000,0,14.917928,3.511448,4,0.00,17.843270,keyphrase,2
9,train_0064,সার্ভিসেস স্পোর্টস কন্ট্রোল বোর্ডের সভাপতি ও স...,তিন বছর,তিন বছর,তিন বছর,1.000000,1,13.445260,3.508925,3,0.00,16.306116,keyphrase,1


fallback predictions loaded: 1500

Confidence thresholds:
light : 13.430680300396531
medium: 15.659253969923844
strict: 16.893610856431195
direct rows: 1500 empty: 0
light rows: 1500 empty: 0
medium rows: 1500 empty: 0
strict rows: 1500 empty: 0

CELL B3 SUBMISSION REPORT
valid_f1 : 0.7018636352660187
valid_em : 0.55
best_params : {'wr_rerank': 0.45, 'wv_vote': 0.5, 'wc_combined': 0.1, 'wt_type': 0.0, 'wl_len': 0.0, 'wp_rank': 0.0, 'valid_f1': 0.7018636352660187, 'valid_em': 0.55}
fallback_path : /content/drive/MyDrive/bangla_qa_070_sprint/sprint3_improved_2_outputs/submission_fresh_gated_light.csv
thresholds : {'light': 13.430680300396531, 'medium': 15.659253969923844, 'strict': 16.893610856431195}
replacements : {'direct': 1500, 'light': 1129, 'medium': 862, 'strict': 620}
fallback_used : {'direct': 0, 'light': 371, 'medium': 638, 'strict': 880}
output_files : {'direct': '/content/drive/MyDrive/bangla_qa_070_sprint/sprint5_reader_ensemble_outputs/submission_b3_direct.csv', 'light': '

In [ ]:
# ============================================================
# B3 No-Fallback Regeneration Cell
# Removes dependency on any previous submission CSV.
#
# Logic:
#   direct = B3 reader/voting answer
#   light/medium/strict = gated reader answer, otherwise direct answer
#
# Outputs:
#   submission_b3_direct_nofallback.csv
#   submission_b3_light_nofallback.csv
#   submission_b3_medium_nofallback.csv
#   submission_b3_strict_nofallback.csv
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json, re
import pandas as pd
import numpy as np

PROJECT_DIR = Path("/content/drive/MyDrive/bangla_qa_070_sprint")
SPRINT_B_DIR = PROJECT_DIR / "sprint5_reader_ensemble_outputs"

TEST_BEST_PATH = SPRINT_B_DIR / "test_best_b3.csv"
VALID_BEST_PATH = SPRINT_B_DIR / "valid_best_b3.csv"

OUT_DIRECT_NF = SPRINT_B_DIR / "submission_b3_direct_nofallback.csv"
OUT_LIGHT_NF = SPRINT_B_DIR / "submission_b3_light_nofallback.csv"
OUT_MEDIUM_NF = SPRINT_B_DIR / "submission_b3_medium_nofallback.csv"
OUT_STRICT_NF = SPRINT_B_DIR / "submission_b3_strict_nofallback.csv"
OUT_REPORT_NF = SPRINT_B_DIR / "cellB3_nofallback_report.json"

assert TEST_BEST_PATH.exists(), TEST_BEST_PATH
assert VALID_BEST_PATH.exists(), VALID_BEST_PATH

# Find test.csv
possible_test_paths = [
    PROJECT_DIR / "test.csv",
    PROJECT_DIR / "data" / "test.csv",
    Path("/content/test.csv"),
    Path("/content/drive/MyDrive/bangla_qa_070_sprint/test.csv"),
]

test_path = next((p for p in possible_test_paths if p.exists()), None)

if test_path is None:
    hits = list(PROJECT_DIR.rglob("test*.csv"))
    test_path = hits[0] if hits else None

assert test_path is not None, "test.csv not found"

test_df = pd.read_csv(test_path)
test_df["index"] = test_df["index"].astype(str)

test_best = pd.read_csv(TEST_BEST_PATH)
valid_best = pd.read_csv(VALID_BEST_PATH)

test_best["index"] = test_best["index"].astype(str)
valid_best["index"] = valid_best["index"].astype(str)

print("test:", test_df.shape)
print("test_best:", test_best.shape)
print("valid_best:", valid_best.shape)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def safe_str(x):
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x)

def clean_answer(ans):
    ans = safe_str(ans)
    ans = ans.replace("\ufeff", " ")
    ans = ans.replace("\u200c", "")
    ans = ans.replace("\u200d", "")
    ans = ans.replace("\xa0", " ")
    ans = ans.replace("–", "-").replace("—", "-")
    ans = re.sub(r"\s+", " ", ans).strip()
    ans = ans.strip(" \"'“”‘’[]{}<>;:,|")
    ans = ans.strip(" .।")
    ans = re.sub(r"\s+", " ", ans).strip()
    return ans

def bad_answer(ans):
    ans = clean_answer(ans)

    if not ans:
        return True

    if len(ans) > 180:
        return True

    bads = [
        "উইকিপিডিয়া",
        "সূচিপত্র",
        "সম্পাদনা",
        "তথ্যসূত্র",
        "পরিভ্রমণ",
        "অজানা যেকোনো পাতা",
        "সাম্প্রতিক পরিবর্তন",
        "কুকির বিবৃতি",
    ]

    return any(b in ans for b in bads)

# ------------------------------------------------------------
# Confidence thresholds from valid_best
# ------------------------------------------------------------

valid_best["confidence"] = pd.to_numeric(valid_best["final_score"], errors="coerce")
q_conf = valid_best["confidence"].quantile([0.20, 0.35, 0.50]).to_dict()

thr_light = float(q_conf[0.20])
thr_medium = float(q_conf[0.35])
thr_strict = float(q_conf[0.50])

print("thresholds:")
print("light :", thr_light)
print("medium:", thr_medium)
print("strict:", thr_strict)

test_best["confidence"] = pd.to_numeric(test_best["final_score"], errors="coerce")
test_best["answer"] = test_best["answer"].map(clean_answer)

best_map = test_best.set_index("index").to_dict("index")

# Direct map from reader/voting output only
direct_map = {}

for idx in test_df["index"].astype(str):
    r = best_map.get(idx, None)

    if r is not None:
        ans = clean_answer(r.get("answer", ""))
        direct_map[idx] = ans
    else:
        direct_map[idx] = ""

# If any direct answer is empty/bad, still use the raw best answer if available.
# No old fallback is used.
empty_direct = sum(1 for x in direct_map.values() if not clean_answer(x))
bad_direct = sum(1 for x in direct_map.values() if bad_answer(x))

print("empty direct:", empty_direct)
print("bad direct:", bad_direct)

def make_nofallback_submission(mode):
    rows = []
    use_reader_count = 0
    use_direct_internal_count = 0
    empty_count = 0

    for idx in test_df["index"].astype(str):
        r = best_map.get(idx, None)

        direct_ans = clean_answer(direct_map.get(idx, ""))
        pred = direct_ans

        if r is not None:
            ans = clean_answer(r.get("answer", ""))
            conf = float(r.get("confidence", -999))

            use_reader = False

            if mode == "direct":
                use_reader = bool(ans)
            elif mode == "light":
                use_reader = bool(ans and conf >= thr_light and not bad_answer(ans))
            elif mode == "medium":
                use_reader = bool(ans and conf >= thr_medium and not bad_answer(ans))
            elif mode == "strict":
                use_reader = bool(ans and conf >= thr_strict and not bad_answer(ans))

            if use_reader:
                pred = ans
                use_reader_count += 1
            else:
                pred = direct_ans
                use_direct_internal_count += 1

        pred = clean_answer(pred)

        if not pred:
            empty_count += 1
            # Last-resort internal no-fallback placeholder.
            # This should almost never happen if B2/B3 covered all test rows.
            pred = "অজানা"

        rows.append({
            "index": str(idx),
            "answer": pred,
        })

    sub = pd.DataFrame(rows)
    return sub, use_reader_count, use_direct_internal_count, empty_count

sub_direct_nf, rep_direct_nf, internal_direct_direct, emp_direct = make_nofallback_submission("direct")
sub_light_nf, rep_light_nf, internal_direct_light, emp_light = make_nofallback_submission("light")
sub_medium_nf, rep_medium_nf, internal_direct_medium, emp_medium = make_nofallback_submission("medium")
sub_strict_nf, rep_strict_nf, internal_direct_strict, emp_strict = make_nofallback_submission("strict")

# ------------------------------------------------------------
# Validate
# ------------------------------------------------------------

def check_submission(sub, name):
    assert len(sub) == len(test_df), f"{name}: wrong row count"
    assert list(sub.columns) == ["index", "answer"], f"{name}: wrong columns"
    assert sub["index"].astype(str).tolist() == test_df["index"].astype(str).tolist(), f"{name}: order mismatch"
    assert sub["answer"].isna().sum() == 0, f"{name}: NaN answer"

    empty = int((sub["answer"].astype(str).str.len() == 0).sum())
    print(name, "rows:", len(sub), "empty:", empty)

check_submission(sub_direct_nf, "direct_nofallback")
check_submission(sub_light_nf, "light_nofallback")
check_submission(sub_medium_nf, "medium_nofallback")
check_submission(sub_strict_nf, "strict_nofallback")

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

sub_direct_nf.to_csv(OUT_DIRECT_NF, index=False, encoding="utf-8-sig")
sub_light_nf.to_csv(OUT_LIGHT_NF, index=False, encoding="utf-8-sig")
sub_medium_nf.to_csv(OUT_MEDIUM_NF, index=False, encoding="utf-8-sig")
sub_strict_nf.to_csv(OUT_STRICT_NF, index=False, encoding="utf-8-sig")

report = {
    "fallback_used": False,
    "old_fallback_csv_used": None,
    "thresholds": {
        "light": thr_light,
        "medium": thr_medium,
        "strict": thr_strict,
    },
    "use_reader_count": {
        "direct": int(rep_direct_nf),
        "light": int(rep_light_nf),
        "medium": int(rep_medium_nf),
        "strict": int(rep_strict_nf),
    },
    "internal_direct_used": {
        "direct": int(internal_direct_direct),
        "light": int(internal_direct_light),
        "medium": int(internal_direct_medium),
        "strict": int(internal_direct_strict),
    },
    "empty_repaired_with_unknown": {
        "direct": int(emp_direct),
        "light": int(emp_light),
        "medium": int(emp_medium),
        "strict": int(emp_strict),
    },
    "outputs": {
        "direct": str(OUT_DIRECT_NF),
        "light": str(OUT_LIGHT_NF),
        "medium": str(OUT_MEDIUM_NF),
        "strict": str(OUT_STRICT_NF),
    },
}

with open(OUT_REPORT_NF, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("\nSaved no-fallback submissions:")
print(OUT_DIRECT_NF)
print(OUT_LIGHT_NF)
print(OUT_MEDIUM_NF)
print(OUT_STRICT_NF)
print(OUT_REPORT_NF)

print("\nReport:")
print(json.dumps(report, ensure_ascii=False, indent=2))

In [ ]:
# ============================================================
# Qwen Integration Cell 1
# Build selective review pack from B3-direct + reader candidates
#
# Base submission: submission_b3_direct.csv
# Qwen will review only selected cases, not all 1500.
#
# Inputs:
#   submission_b3_direct.csv
#   submission_b3_light.csv
#   test_best_b3.csv
#   test_reader_candidates_v2.parquet
#   test_reranked_v2.parquet
#
# Output:
#   qwen_selective_review_pack.parquet
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import re, json, time, gc
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

t0 = time.time()

PROJECT_DIR = Path("/content/drive/MyDrive/bangla_qa_070_sprint")
SPRINT_A_DIR = PROJECT_DIR / "sprint4_evidence_recall_outputs"
SPRINT_B_DIR = PROJECT_DIR / "sprint5_reader_ensemble_outputs"

DIRECT_PATH = SPRINT_B_DIR / "submission_b3_direct_nofallback.csv"
LIGHT_PATH = SPRINT_B_DIR / "submission_b3_light_nofallback.csv"

OUT_PACK = SPRINT_B_DIR / "qwen_selective_review_pack_nofallback.parquet"
OUT_DEBUG = SPRINT_B_DIR / "qwen_selective_review_pack_nofallback_debug.csv"

#DIRECT_PATH = SPRINT_B_DIR / "submission_b3_direct.csv"
#LIGHT_PATH = SPRINT_B_DIR / "submission_b3_light.csv"

TEST_BEST_PATH = SPRINT_B_DIR / "test_best_b3.csv"
TEST_CAND_PATH = SPRINT_B_DIR / "test_reader_candidates_v2.parquet"
TEST_RERANKED_PATH = SPRINT_A_DIR / "test_reranked_v2.parquet"

#OUT_PACK = SPRINT_B_DIR / "qwen_selective_review_pack.parquet"
#OUT_DEBUG = SPRINT_B_DIR / "qwen_selective_review_pack_debug.csv"

for p in [DIRECT_PATH, LIGHT_PATH, TEST_BEST_PATH, TEST_CAND_PATH, TEST_RERANKED_PATH]:
    assert p.exists(), f"Missing: {p}"

print("direct:", DIRECT_PATH)
print("light:", LIGHT_PATH)
print("test_best:", TEST_BEST_PATH)
print("reader candidates:", TEST_CAND_PATH)
print("reranked:", TEST_RERANKED_PATH)

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------

MAX_REVIEW = 400          # keep Qwen cost/time reasonable
TOP_ANSWERS = 8
TOP_EVIDENCE = 3

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

BN_DIGITS = "০১২৩৪৫৬৭৮৯"
EN_DIGITS = "0123456789"
BN_TO_EN = str.maketrans(BN_DIGITS, EN_DIGITS)

def safe_str(x):
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x)

def normalize_bn_text(x):
    x = safe_str(x)
    x = x.replace("\ufeff", " ")
    x = x.replace("\u200c", "")
    x = x.replace("\u200d", "")
    x = x.replace("\xa0", " ")
    x = x.replace("–", "-").replace("—", "-")
    x = x.replace("“", "\"").replace("”", "\"")
    x = x.replace("‘", "'").replace("’", "'")
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def clean_answer(ans):
    ans = normalize_bn_text(ans)
    ans = ans.strip(" \"'“”‘’[]{}<>;:,|")
    ans = ans.strip(" .।")
    ans = re.sub(r"\s+", " ", ans).strip()
    return ans

def norm_ans(x):
    x = clean_answer(x).lower()
    x = x.translate(BN_TO_EN)
    x = re.sub(r"[।,;:!?\"'“”‘’()\[\]{}<>|/\\+=*_~`\-–—]", " ", x)
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def strip_case_suffix(ans):
    ans = clean_answer(ans)
    if re.fullmatch(r"[০-৯0-9,./-]+", ans):
        return ans
    if len(ans.split()) > 4:
        return ans

    for pat in [r"কে$", r"তে$", r"ের$", r"র$", r"য়$", r"য়$", r"ে$"]:
        cand = re.sub(pat, "", ans).strip()
        if len(cand) >= 2 and cand != ans:
            return clean_answer(cand)
    return ans

def postprocess_answer(ans, question=""):
    ans = clean_answer(ans)

    ans = ans.replace("মনামাকে", "মনামা")
    ans = ans.replace("লাহোরে", "লাহোর")
    ans = ans.replace("পুনের", "পুনে")

    q = normalize_bn_text(question)
    if re.search(r"(কোন শহর|কোন দেশ|কোথায়|কোথায়|কে|কার নাম|নাম কী|নাম কি)", q):
        ans = strip_case_suffix(ans)

    ans = re.sub(r"^(কোন|কে|কি|কী|কবে|কত|কোথায়|কোথায়)\s+", "", ans).strip()
    ans = re.sub(r"\s+(করে|করেন|ছিল|ছিলেন|হয়|হয়|হলো|হল)$", "", ans).strip()
    ans = re.sub(r"\s*,\s*", ", ", ans)
    ans = re.sub(r"\s+", " ", ans).strip()

    return clean_answer(ans)

def bad_answer(ans):
    ans = clean_answer(ans)
    if not ans:
        return True
    if len(ans) > 170:
        return True
    if len(ans) <= 1 and not re.search(r"[০-৯0-9]", ans):
        return True

    bads = [
        "উইকিপিডিয়া",
        "সূচিপত্র",
        "সম্পাদনা",
        "তথ্যসূত্র",
        "পরিভ্রমণ",
        "অজানা যেকোনো পাতা",
        "সাম্প্রতিক পরিবর্তন",
        "এই পাতাটি",
        "কুকির বিবৃতি",
    ]
    return any(b in ans for b in bads)

def qtype(question):
    q = normalize_bn_text(question)
    if re.search(r"(কত সালে|কোন সালে|সাল|বছর|খ্রিস্টাব্দ|খ্রীস্টাব্দ)", q):
        return "year"
    if re.search(r"(কবে|তারিখ|জন্ম|মৃত্যু)", q):
        return "date"
    if re.search(r"(কত|কয়টি|কতটি|কতজন|সংখ্যা|দৈর্ঘ্য|উচ্চতা|আয়তন)", q):
        return "number"
    if re.search(r"(কে|কার|ব্যক্তি|লেখক|পরিচালক|প্রতিষ্ঠাতা|সভাপতি|অধিনায়ক|মন্ত্রী)", q):
        return "person"
    if re.search(r"(কোথায়|কোথায়|কোন দেশে|কোন শহর|স্থান|রাজ্য|জেলা)", q):
        return "place"
    return "other"

def answer_type_bonus(ans, question):
    qt = qtype(question)
    ans = clean_answer(ans)

    if qt == "year":
        if re.fullmatch(r"[০-৯0-9]{3,4}", ans):
            return 1.0
        if re.search(r"[০-৯0-9]{3,4}", ans):
            return 0.4
    if qt == "number" and re.search(r"[০-৯0-9]", ans):
        return 0.6
    if qt == "place" and len(ans.split()) <= 4:
        return 0.3
    if qt == "person" and len(ans.split()) <= 5:
        return 0.25
    return 0.0

def compact_window(w, max_chars=850):
    w = normalize_bn_text(w)
    bad_fragments = [
        "পরিভ্রমণ প্রধান পাতা সম্প্রদায়ের প্রবেশদ্বার",
        "সাম্প্রতিক পরিবর্তন অজানা যেকোনো পাতা সাহায্য",
        "উইকিপিডিয়া® একটি অলাভজনক সংস্থা",
        "আচরণবিধি উন্নয়নকারী পরিসংখ্যান কুকির বিবৃতি",
        "সূচিপত্র টগল করুন",
    ]
    for b in bad_fragments:
        w = w.replace(b, " ")
    w = re.sub(r"\s+", " ", w).strip()
    return w[:max_chars]

# ------------------------------------------------------------
# Load artifacts
# ------------------------------------------------------------

direct = pd.read_csv(DIRECT_PATH)
light = pd.read_csv(LIGHT_PATH)
test_best = pd.read_csv(TEST_BEST_PATH)
cand = pd.read_parquet(TEST_CAND_PATH)
reranked = pd.read_parquet(TEST_RERANKED_PATH)

for df in [direct, light, test_best, cand, reranked]:
    df["index"] = df["index"].astype(str)

direct = direct.rename(columns={"answer": "direct_answer"})
light = light.rename(columns={"answer": "light_answer"})

base = direct.merge(light[["index", "light_answer"]], on="index", how="left")

# Add test_best confidence/meta
meta_cols = [c for c in [
    "index", "question", "answer", "answer_raw", "final_score",
    "reader_score", "rerank_score", "vote_count",
    "type_bonus", "source", "chunk_rank"
] if c in test_best.columns]

base = base.merge(test_best[meta_cols], on="index", how="left", suffixes=("", "_best"))

if "question" not in base.columns:
    # fallback from cand
    qmap = cand.drop_duplicates("index").set_index("index")["question"].to_dict()
    base["question"] = base["index"].map(qmap)

base["direct_answer"] = base["direct_answer"].map(clean_answer)
base["light_answer"] = base["light_answer"].map(clean_answer)
base["question"] = base["question"].map(normalize_bn_text)

print("base:", base.shape)
print("cand:", cand.shape)
print("reranked:", reranked.shape)

# ------------------------------------------------------------
# Clean candidate answers
# ------------------------------------------------------------

cand["answer_raw"] = cand["answer"].astype(str)
cand["answer"] = cand.apply(lambda r: postprocess_answer(r["answer_raw"], r["question"]), axis=1)
cand["answer_norm"] = cand["answer"].map(norm_ans)
cand["bad"] = cand["answer"].map(bad_answer)
cand["type_bonus"] = cand.apply(lambda r: answer_type_bonus(r["answer"], r["question"]), axis=1)
cand["answer_word_count"] = cand["answer"].map(lambda x: len(str(x).split()))

cand = cand[cand["bad"] == False].copy()

# ------------------------------------------------------------
# Build aggregate candidate stats
# ------------------------------------------------------------

agg = (
    cand.groupby(["index", "answer_norm"])
    .agg(
        answer=("answer", "first"),
        vote_count=("answer", "count"),
        best_reader=("reader_score", "max"),
        mean_reader=("reader_score", "mean"),
        best_rerank=("rerank_score", "max"),
        best_combined=("combined_score_v2", "max"),
        min_chunk_rank=("chunk_rank", "min"),
        source_count=("source", "nunique"),
        type_bonus=("type_bonus", "max"),
        answer_word_count=("answer_word_count", "min"),
    )
    .reset_index()
)

agg["selector_score"] = (
    agg["best_reader"].astype(float)
    + 0.35 * agg["best_rerank"].astype(float)
    + 0.25 * np.log1p(agg["vote_count"].astype(float))
    + 0.08 * agg["best_combined"].astype(float)
    + 0.60 * agg["type_bonus"].astype(float)
    - 0.015 * agg["min_chunk_rank"].astype(float)
    - 0.04 * np.maximum(0, agg["answer_word_count"].astype(float) - 10)
)

# top2 margin per q
tmp = agg.sort_values(["index", "selector_score"], ascending=[True, False]).copy()
tmp["rank"] = tmp.groupby("index").cumcount() + 1
top1 = tmp[tmp["rank"] == 1][["index", "answer", "answer_norm", "selector_score", "vote_count"]].rename(
    columns={
        "answer": "top1_answer",
        "answer_norm": "top1_norm",
        "selector_score": "top1_score",
        "vote_count": "top1_vote_count",
    }
)
top2 = tmp[tmp["rank"] == 2][["index", "selector_score"]].rename(columns={"selector_score": "top2_score"})

base = base.merge(top1, on="index", how="left").merge(top2, on="index", how="left")

base["top2_score"] = base["top2_score"].fillna(-999)
base["score_margin"] = base["top1_score"].fillna(-999) - base["top2_score"].fillna(-999)

base["direct_norm"] = base["direct_answer"].map(norm_ans)
base["light_norm"] = base["light_answer"].map(norm_ans)
base["top1_norm"] = base["top1_norm"].fillna("")

base["direct_light_diff"] = base["direct_norm"] != base["light_norm"]
base["direct_not_top1"] = base["direct_norm"] != base["top1_norm"]

# direct confidence if available
if "final_score" in base.columns:
    base["final_score"] = pd.to_numeric(base["final_score"], errors="coerce")
else:
    base["final_score"] = np.nan

conf_q35 = float(base["final_score"].quantile(0.35)) if base["final_score"].notna().any() else -999
conf_q60 = float(base["final_score"].quantile(0.60)) if base["final_score"].notna().any() else -999

# ------------------------------------------------------------
# Select review rows
# ------------------------------------------------------------

def review_reason(row):
    reasons = []

    if row["direct_light_diff"]:
        reasons.append("direct_light_diff")

    if row["direct_not_top1"] and row.get("top1_vote_count", 0) >= 2:
        reasons.append("reader_top1_diff_vote")

    if row["score_margin"] <= 0.8:
        reasons.append("close_candidates")

    if pd.notna(row["final_score"]) and row["final_score"] < conf_q35:
        reasons.append("low_direct_conf")

    qt = qtype(row["question"])
    if qt in ["year", "number", "place", "person"]:
        reasons.append("short_fact_type")

    return "|".join(reasons)

base["review_reason"] = base.apply(review_reason, axis=1)

# priority: likely useful, not all rows
base["review_priority"] = (
    3.0 * base["direct_light_diff"].astype(float)
    + 2.0 * base["direct_not_top1"].astype(float)
    + 1.2 * (base["score_margin"] <= 0.8).astype(float)
    + 1.0 * (base["final_score"].fillna(999) < conf_q35).astype(float)
    + 0.8 * base["question"].map(lambda q: qtype(q) in ["year", "number", "place", "person"]).astype(float)
    + 0.4 * np.log1p(base["top1_vote_count"].fillna(0).astype(float))
)

review_base = base[
    (base["review_reason"].str.len() > 0)
    & (base["top1_answer"].notna())
].copy()

review_base = review_base.sort_values("review_priority", ascending=False).head(MAX_REVIEW).reset_index(drop=True)

print("selected review rows:", review_base.shape)

# ------------------------------------------------------------
# Build prompt pack
# ------------------------------------------------------------

cand_groups = {k: g.copy() for k, g in agg.groupby("index")}
rerank_groups = {k: g.copy() for k, g in reranked.groupby("index")}

def build_pack_row(row):
    qid = str(row["index"])
    question = normalize_bn_text(row["question"])
    direct_ans = clean_answer(row["direct_answer"])
    light_ans = clean_answer(row["light_answer"])

    cg = cand_groups.get(qid, pd.DataFrame(columns=agg.columns)).copy()
    rg = rerank_groups.get(qid, pd.DataFrame(columns=reranked.columns)).copy()

    if len(cg):
        cg = cg.sort_values(["selector_score", "vote_count", "best_reader"], ascending=False).head(TOP_ANSWERS).copy()
    else:
        cg = pd.DataFrame(columns=agg.columns)

    # Make sure direct/light are in candidate list when possible
    existing_norms = set(cg["answer_norm"].astype(str).tolist()) if len(cg) else set()
    extra = []

    for special_name, ans in [("b3_direct", direct_ans), ("b3_light", light_ans)]:
        n = norm_ans(ans)
        if ans and n and n not in existing_norms:
            extra.append({
                "answer_norm": n,
                "answer": ans,
                "vote_count": 0,
                "best_reader": 0.0,
                "best_rerank": 0.0,
                "best_combined": 0.0,
                "min_chunk_rank": 999,
                "source_count": 0,
                "type_bonus": answer_type_bonus(ans, question),
                "answer_word_count": len(ans.split()),
                "selector_score": -1.0,
                "special": special_name,
            })
            existing_norms.add(n)

    if extra:
        cg = pd.concat([cg, pd.DataFrame(extra)], ignore_index=True)

    cg = cg.sort_values(["selector_score", "vote_count", "best_reader"], ascending=False).head(TOP_ANSWERS).reset_index(drop=True)

    labels = list("ABCDEFGH")
    candidates = []

    for i, r in cg.iterrows():
        candidates.append({
            "label": labels[i],
            "answer": clean_answer(r["answer"]),
            "score": float(r.get("selector_score", 0)),
            "vote_count": int(r.get("vote_count", 0)),
            "best_reader": float(r.get("best_reader", 0)),
            "best_rerank": float(r.get("best_rerank", 0)),
            "min_chunk_rank": int(r.get("min_chunk_rank", 999)),
        })

    ev = rg.sort_values("combined_score_v2", ascending=False).head(TOP_EVIDENCE).copy()
    evidence = []

    for i, r in enumerate(ev.itertuples(index=False), start=1):
        evidence.append({
            "evidence_id": i,
            "source": getattr(r, "source", ""),
            "chunk_rank": int(getattr(r, "chunk_rank", -1)),
            "text": compact_window(getattr(r, "window", ""), 850),
        })

    return {
        "index": qid,
        "question": question,
        "direct_answer": direct_ans,
        "light_answer": light_ans,
        "top_candidate": candidates[0]["answer"] if candidates else direct_ans,
        "review_reason": row["review_reason"],
        "review_priority": float(row["review_priority"]),
        "candidates_json": json.dumps(candidates, ensure_ascii=False),
        "evidence_json": json.dumps(evidence, ensure_ascii=False),
    }

pack_rows = [build_pack_row(r) for r in review_base.to_dict("records")]
pack_df = pd.DataFrame(pack_rows)

pack_df.to_parquet(OUT_PACK, index=False)
pack_df.to_csv(OUT_DEBUG, index=False, encoding="utf-8-sig")

print("\nSaved:")
print(OUT_PACK)
print(OUT_DEBUG)

print("\nReview reason counts:")
display(pack_df["review_reason"].value_counts().head(20).to_frame("count"))

print("\nPreview:")
display(pack_df.head(10))

gc.collect()
print("\n✅ Qwen Cell 1 complete. Runtime:", round(time.time() - t0, 2), "sec")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
direct: /content/drive/MyDrive/bangla_qa_070_sprint/sprint5_reader_ensemble_outputs/submission_b3_direct.csv
light: /content/drive/MyDrive/bangla_qa_070_sprint/sprint5_reader_ensemble_outputs/submission_b3_light.csv
test_best: /content/drive/MyDrive/bangla_qa_070_sprint/sprint5_reader_ensemble_outputs/test_best_b3.csv
reader candidates: /content/drive/MyDrive/bangla_qa_070_sprint/sprint5_reader_ensemble_outputs/test_reader_candidates_v2.parquet
reranked: /content/drive/MyDrive/bangla_qa_070_sprint/sprint4_evidence_recall_outputs/test_reranked_v2.parquet
base: (1500, 13)
cand: (1285764, 24)
reranked: (30000, 19)
selected review rows: (400, 25)

Saved:
/content/drive/MyDrive/bangla_qa_070_sprint/sprint5_reader_ensemble_outputs/qwen_selective_review_pack.parquet
/content/drive/MyDrive/bangla_qa_070_sprint/sprint5_reader_ensemble_outputs/qwen_selective_review_pac

,count
review_reason,
direct_light_diff|close_candidates|low_direct_conf,76
direct_light_diff|low_direct_conf|short_fact_type,59
direct_light_diff|low_direct_conf,58
direct_light_diff|close_candidates|low_direct_conf|short_fact_type,51
close_candidates|low_direct_conf|short_fact_type,37
reader_top1_diff_vote|short_fact_type,20
direct_light_diff|reader_top1_diff_vote|close_candidates|low_direct_conf,15
direct_light_diff|reader_top1_diff_vote|close_candidates|low_direct_conf|short_fact_type,14
close_candidates|low_direct_conf,12



Preview:


,index,question,direct_answer,light_answer,top_candidate,review_reason,review_priority,candidates_json,evidence_json
0,test_0479,২০২৪ সালে ধর্ম সম্পর্কিত ঘটনাগুলির সময়রেখা কো...,২০২৪ সালের,২০২৩,২০২৪,direct_light_diff|reader_top1_diff_vote|close_...,8.831777,"[{""label"": ""A"", ""answer"": ""২০২৪"", ""score"": 6.7...","[{""evidence_id"": 1, ""source"": ""keyphrase"", ""ch..."
1,test_0650,১৯৫৪ কলম্বো কাপে অংশগ্রহণকারী দলগুলোর মধ্যে কো...,প্যারাগুয়ে,রাজস্থান ইউনাইটেড,ভারত,direct_light_diff|reader_top1_diff_vote|close_...,8.778364,"[{""label"": ""A"", ""answer"": ""ভারত"", ""score"": 9.2...","[{""evidence_id"": 1, ""source"": ""keyphrase"", ""ch..."
2,test_0079,পাত্রিয়া আমাদা কোন জাতীয় সঙ্গীতকে প্রতিস্থাপ...,মোজাম্বিকে,ফ্রেলিমো,মোজাম্বিক,direct_light_diff|reader_top1_diff_vote|close_...,8.778364,"[{""label"": ""A"", ""answer"": ""মোজাম্বিক"", ""score""...","[{""evidence_id"": 1, ""source"": ""keyphrase"", ""ch..."
3,test_0489,উজবেকিস্তানে বহুবিবাহ সম্পর্কে কী ধরনের বিতর্ক...,বহুবিবাহকে বৈধ করার বিষয়েও বিতর্ক,বেআইনি,বেআইনি,direct_light_diff|reader_top1_diff_vote|close_...,8.716704,"[{""label"": ""A"", ""answer"": ""বেআইনি"", ""score"": 1...","[{""evidence_id"": 1, ""source"": ""keyphrase"", ""ch..."
4,test_0694,ভারতীয় সংখ্যা পদ্ধতি অনুযায়ী ১০-১৫ থেকে ১০১৫...,ভারতীয় সংখ্যা পদ্ধতি অনুযায়ী নাম,সংখ্যা পদ্ধতির ভিত্তি,ভারতীয় সংখ্যা পদ্ধতি অনুযায়ী,direct_light_diff|reader_top1_diff_vote|close_...,8.643775,"[{""label"": ""A"", ""answer"": ""ভারতীয় সংখ্যা পদ্ধ...","[{""evidence_id"": 1, ""source"": ""keyphrase"", ""ch..."
5,test_0847,মিডিয়া পণ্ডিত লিভ ম্যানোভিচ কী বলেছেন এসব আর্...,ব্যবহারকারীদের অনেক উপায়ে উপকরণ নেভিগেট করতে ...,Previous color television,ডাটাবেস ফরম হিসেবে সাহায্য করবে,direct_light_diff|reader_top1_diff_vote|close_...,8.643775,"[{""label"": ""A"", ""answer"": ""ডাটাবেস ফরম হিসেবে ...","[{""evidence_id"": 1, ""source"": ""keyphrase"", ""ch..."
6,test_0432,বাংলাদেশের পাট শিল্পে উৎপাদিত দ্রব্যের মধ্যে ক...,উতপাদিত দ্রব্য,সোনালী ব্যাগ,"সুতা, দড়ি, চট, বস্তা, চা, ব্যাগ, মোড়ক, তন্তু",direct_light_diff|reader_top1_diff_vote|close_...,8.643775,"[{""label"": ""A"", ""answer"": ""সুতা, দড়ি, চট, বস্...","[{""evidence_id"": 1, ""source"": ""keyphrase"", ""ch..."
7,test_0414,ভারতে প্রচারিত বিজ্ঞাপনের কারণে সৃষ্ট বিতর্কের...,ভারত এবং ভারতীয় সম্প্রদায়ের বিতর্কিত বিজ্ঞাপন,লাভ-জিহাদ,বিতর্কিত বিজ্ঞাপন,direct_light_diff|reader_top1_diff_vote|close_...,8.643775,"[{""label"": ""A"", ""answer"": ""বিতর্কিত বিজ্ঞাপন"",...","[{""evidence_id"": 1, ""source"": ""keyphrase"", ""ch..."
8,test_0440,কারা-খোজা নামে বিকেন্দ্রীকৃত বৌদ্ধ রাজ্যগুলোর ...,উইঘু,উইঘুর সাম্রাজ্য,কারা-খানলিক,direct_light_diff|reader_top1_diff_vote|close_...,8.554518,"[{""label"": ""A"", ""answer"": ""কারা-খানলিক"", ""scor...","[{""evidence_id"": 1, ""source"": ""keyphrase"", ""ch..."
9,test_1157,৫৬তম ফিল্মফেয়ার পুরস্কারে শ্রেষ্ঠ পার্শ্ব অভি...,খিচড়ি: দ্য মুভি,উই আর ফ্যামিলি,গোলমাল থ্রি ও খিচড়ি: দ্য মুভি,direct_light_diff|reader_top1_diff_vote|close_...,8.554518,"[{""label"": ""A"", ""answer"": ""গোলমাল থ্রি ও খিচড়...","[{""evidence_id"": 1, ""source"": ""keyphrase"", ""ch..."



✅ Qwen Cell 1 complete. Runtime: 155.38 sec


In [ ]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.9 MB/s eta 0:00:00


In [ ]:
# ============================================================
# Patch: Fix bitsandbytes 4-bit quantization version issue
# ============================================================

!pip -q uninstall -y bitsandbytes
!pip -q install -U "bitsandbytes>=0.46.1" "accelerate>=0.26.0" "transformers>=4.45.0"

import bitsandbytes as bnb
import transformers
import accelerate
import torch

print("bitsandbytes:", bnb.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

print("\n✅ Patch complete.")
print("Now restart runtime if needed, then rerun Qwen Cell 2.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 59.6 MB/s eta 0:00:00
bitsandbytes: 0.49.2
transformers: 5.0.0
accelerate: 1.13.0
torch: 2.11.0+cu128
cuda available: True
gpu: Tesla T4

✅ Patch complete.
Now restart runtime if needed, then rerun Qwen Cell 2.


In [ ]:
# ============================================================
# Qwen Integration Cell 2
# Qwen2.5-7B-Instruct selective correction over B3-direct
#
# Outputs:
#   submission_b3_direct_qwen_cautious.csv
#   submission_b3_direct_qwen_medium.csv
#   submission_b3_direct_qwen_aggressive.csv
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import re, json, time, gc
from pathlib import Path

import numpy as np
import pandas as pd

t0 = time.time()

try:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
except Exception:
    !pip -q install -U transformers accelerate bitsandbytes
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

PROJECT_DIR = Path("/content/drive/MyDrive/bangla_qa_070_sprint")
SPRINT_B_DIR = PROJECT_DIR / "sprint5_reader_ensemble_outputs"



#PACK_PATH = SPRINT_B_DIR / "qwen_selective_review_pack.parquet"
#DIRECT_PATH = SPRINT_B_DIR / "submission_b3_direct.csv"

#OUT_RAW = SPRINT_B_DIR / "qwen_selective_raw_outputs.csv"
#OUT_CAUTIOUS = SPRINT_B_DIR / "submission_b3_direct_qwen_cautious.csv"
#OUT_MEDIUM = SPRINT_B_DIR / "submission_b3_direct_qwen_medium.csv"
#OUT_AGGRESSIVE = SPRINT_B_DIR / "submission_b3_direct_qwen_aggressive.csv"
#OUT_REPORT = SPRINT_B_DIR / "qwen_selective_report.json"

PACK_PATH = SPRINT_B_DIR / "qwen_selective_review_pack_nofallback.parquet"
DIRECT_PATH = SPRINT_B_DIR / "submission_b3_direct_nofallback.csv"

OUT_RAW = SPRINT_B_DIR / "qwen_selective_raw_outputs_nofallback.csv"
OUT_CAUTIOUS = SPRINT_B_DIR / "submission_b3_direct_qwen_cautious_nofallback.csv"
OUT_MEDIUM = SPRINT_B_DIR / "submission_b3_direct_qwen_medium_nofallback.csv"
OUT_AGGRESSIVE = SPRINT_B_DIR / "submission_b3_direct_qwen_aggressive_nofallback.csv"
OUT_REPORT = SPRINT_B_DIR / "qwen_selective_report_nofallback.json"

assert PACK_PATH.exists(), PACK_PATH
assert DIRECT_PATH.exists(), DIRECT_PATH

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
MAX_NEW_TOKENS = 32
SAVE_EVERY = 25

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("⚠️ CPU will be too slow. Use GPU.")

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

BN_DIGITS = "০১২৩৪৫৬৭৮৯"
EN_DIGITS = "0123456789"
BN_TO_EN = str.maketrans(BN_DIGITS, EN_DIGITS)

def safe_str(x):
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x)

def normalize_bn_text(x):
    x = safe_str(x)
    x = x.replace("\ufeff", " ")
    x = x.replace("\u200c", "")
    x = x.replace("\u200d", "")
    x = x.replace("\xa0", " ")
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def clean_answer(ans):
    ans = normalize_bn_text(ans)
    ans = ans.strip(" \"'“”‘’[]{}<>;:,|")
    ans = ans.strip(" .।")
    ans = re.sub(r"\s+", " ", ans).strip()
    return ans

def norm_ans(x):
    x = clean_answer(x).lower()
    x = x.translate(BN_TO_EN)
    x = re.sub(r"[।,;:!?\"'“”‘’()\[\]{}<>|/\\+=*_~`\-–—]", " ", x)
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def parse_json_cell(x):
    if isinstance(x, list):
        return x
    return json.loads(str(x))

def extract_qwen_answer(text):
    text = normalize_bn_text(text)
    text = re.sub(r"^(উত্তর|Answer|Final Answer|Selected Answer)\s*[:：-]\s*", "", text, flags=re.I).strip()
    text = text.split("\n")[0].strip()
    text = re.sub(r"^[A-Z]\s*[\.\):：-]\s*", "", text).strip()
    return clean_answer(text)

def choose_valid_answer(qwen_text, candidates):
    qwen_ans = extract_qwen_answer(qwen_text)
    qn = norm_ans(qwen_ans)

    cand_answers = [clean_answer(c["answer"]) for c in candidates]
    cand_norms = [norm_ans(a) for a in cand_answers]

    # exact candidate answer
    for a, n in zip(cand_answers, cand_norms):
        if qn and qn == n:
            return a, "qwen_exact"

    # label only
    raw = clean_answer(qwen_text).upper().strip()
    for c in candidates:
        if raw == str(c["label"]).upper():
            return clean_answer(c["answer"]), "qwen_label"

    # label prefix or soft match
    for c, a, n in zip(candidates, cand_answers, cand_norms):
        label = str(c["label"]).upper()
        if raw.startswith(label):
            return a, "qwen_label_prefix"
        if n and qn and (n in qn or qn in n) and min(len(n), len(qn)) >= 2:
            return a, "qwen_soft_match"

    return "", "invalid"

def build_prompt(question, direct_answer, light_answer, candidates, evidence):
    cand_lines = []
    for c in candidates:
        cand_lines.append(f"{c['label']}. {clean_answer(c['answer'])}")

    ev_lines = []
    for e in evidence:
        txt = normalize_bn_text(e.get("text", ""))
        ev_lines.append(f"[Evidence {e.get('evidence_id', '')}] {txt}")

    prompt = f"""তুমি একটি বাংলা QA answer selector.

নিয়ম:
- Candidate Answers থেকে ঠিক একটি উত্তর বেছে নাও।
- নিজের থেকে নতুন উত্তর বানাবে না।
- ব্যাখ্যা দেবে না।
- শুধু নির্বাচিত candidate answer text লিখবে।
- যদি B3 Direct Answer যথেষ্ট ভালো হয়, সেটিই বেছে নাও।
- উত্তর ছোট ও exact span হওয়া উচিত।

Question:
{question}

B3 Direct Answer:
{direct_answer}

B3 Light Answer:
{light_answer}

Evidence:
{chr(10).join(ev_lines)}

Candidate Answers:
{chr(10).join(cand_lines)}

Selected Answer:"""

    return prompt

# ------------------------------------------------------------
# Load pack and base
# ------------------------------------------------------------

pack = pd.read_parquet(PACK_PATH)
pack["index"] = pack["index"].astype(str)

base = pd.read_csv(DIRECT_PATH)
base["index"] = base["index"].astype(str)
base["answer"] = base["answer"].map(clean_answer)

print("review pack:", pack.shape)
print("base direct:", base.shape)

# Resume support
done = {}
if OUT_RAW.exists():
    old = pd.read_csv(OUT_RAW)
    if "index" in old.columns and "selected_answer" in old.columns:
        done = dict(zip(old["index"].astype(str), old.to_dict("records")))
        print("resume rows:", len(done))

# ------------------------------------------------------------
# Load Qwen 7B 4-bit
# ------------------------------------------------------------

print("\nLoading Qwen2.5-7B-Instruct 4-bit...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.eval()
print("Qwen loaded.")

# ------------------------------------------------------------
# Generate Qwen selections
# ------------------------------------------------------------

rows = list(done.values())
done_ids = set(done.keys())

for i, r in enumerate(pack.itertuples(index=False), start=1):
    qid = str(getattr(r, "index"))

    if qid in done_ids:
        continue

    question = normalize_bn_text(getattr(r, "question"))
    direct_answer = clean_answer(getattr(r, "direct_answer"))
    light_answer = clean_answer(getattr(r, "light_answer"))

    candidates = parse_json_cell(getattr(r, "candidates_json"))
    evidence = parse_json_cell(getattr(r, "evidence_json"))

    prompt = build_prompt(question, direct_answer, light_answer, candidates, evidence)

    messages = [
        {
            "role": "system",
            "content": "You are a strict Bangla extractive QA answer selector. Output only one candidate answer text.",
        },
        {
            "role": "user",
            "content": prompt,
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    gen_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    raw_out = tokenizer.decode(gen_ids, skip_special_tokens=True)

    selected, select_mode = choose_valid_answer(raw_out, candidates)

    direct_norm = norm_ans(direct_answer)
    selected_norm = norm_ans(selected)

    rows.append({
        "index": qid,
        "question": question,
        "direct_answer": direct_answer,
        "light_answer": light_answer,
        "selected_answer": clean_answer(selected),
        "qwen_raw": raw_out,
        "select_mode": select_mode,
        "changed_from_direct": bool(selected_norm and selected_norm != direct_norm),
        "review_reason": getattr(r, "review_reason"),
        "review_priority": float(getattr(r, "review_priority")),
    })

    if i % SAVE_EVERY == 0:
        raw_df = pd.DataFrame(rows)
        raw_df.to_csv(OUT_RAW, index=False, encoding="utf-8-sig")
        print(f"processed {i}/{len(pack)} | saved {len(raw_df)}")

raw_df = pd.DataFrame(rows)
raw_df.to_csv(OUT_RAW, index=False, encoding="utf-8-sig")

print("\nQwen selection counts:")
display(raw_df["select_mode"].value_counts().to_frame("count"))
print("changed_from_direct:", int(raw_df["changed_from_direct"].sum()))

# ------------------------------------------------------------
# Merge strategies
# ------------------------------------------------------------

qwen_map = raw_df.set_index("index").to_dict("index")

def make_submission(mode):
    sub = base.copy()
    changed = 0

    new_answers = []

    for r in sub.itertuples(index=False):
        qid = str(getattr(r, "index"))
        base_ans = clean_answer(getattr(r, "answer"))

        q = qwen_map.get(qid, None)
        final = base_ans

        if q is not None:
            selected = clean_answer(q.get("selected_answer", ""))
            select_mode = str(q.get("select_mode", ""))
            changed_from_direct = bool(q.get("changed_from_direct", False))
            priority = float(q.get("review_priority", 0))

            valid_mode = select_mode in {
                "qwen_exact",
                "qwen_label",
                "qwen_label_prefix",
                "qwen_soft_match",
            }

            if selected and valid_mode and changed_from_direct:
                if mode == "aggressive":
                    final = selected
                elif mode == "medium":
                    if select_mode in {"qwen_exact", "qwen_label", "qwen_label_prefix"} and priority >= 2.0:
                        final = selected
                elif mode == "cautious":
                    if select_mode in {"qwen_exact", "qwen_label"} and priority >= 3.0:
                        final = selected

        if norm_ans(final) != norm_ans(base_ans):
            changed += 1

        new_answers.append(clean_answer(final))

    sub["answer"] = new_answers
    return sub, changed

sub_cautious, ch_cautious = make_submission("cautious")
sub_medium, ch_medium = make_submission("medium")
sub_aggressive, ch_aggressive = make_submission("aggressive")

for name, sub in [
    ("cautious", sub_cautious),
    ("medium", sub_medium),
    ("aggressive", sub_aggressive),
]:
    assert len(sub) == len(base)
    assert list(sub.columns) == ["index", "answer"]
    assert sub["index"].astype(str).tolist() == base["index"].astype(str).tolist()
    assert sub["answer"].isna().sum() == 0
    print(name, "rows:", len(sub), "empty:", int((sub["answer"].astype(str).str.len() == 0).sum()))

sub_cautious.to_csv(OUT_CAUTIOUS, index=False, encoding="utf-8-sig")
sub_medium.to_csv(OUT_MEDIUM, index=False, encoding="utf-8-sig")
sub_aggressive.to_csv(OUT_AGGRESSIVE, index=False, encoding="utf-8-sig")

report = {
    "base": str(DIRECT_PATH),
    "review_pack": str(PACK_PATH),
    "model": MODEL_NAME,
    "review_rows": int(len(pack)),
    "qwen_rows": int(len(raw_df)),
    "select_mode_counts": {str(k): int(v) for k, v in raw_df["select_mode"].value_counts().items()},
    "changed_from_direct_raw": int(raw_df["changed_from_direct"].sum()),
    "submission_changes": {
        "cautious": int(ch_cautious),
        "medium": int(ch_medium),
        "aggressive": int(ch_aggressive),
    },
    "outputs": {
        "cautious": str(OUT_CAUTIOUS),
        "medium": str(OUT_MEDIUM),
        "aggressive": str(OUT_AGGRESSIVE),
        "raw": str(OUT_RAW),
    },
    "runtime_sec": round(time.time() - t0, 2),
}

with open(OUT_REPORT, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("\n" + "=" * 70)
print("QWEN SELECTIVE REPORT")
print("=" * 70)
for k, v in report.items():
    print(k, ":", v)

print("\nSaved:")
print(OUT_CAUTIOUS)
print(OUT_MEDIUM)
print(OUT_AGGRESSIVE)
print(OUT_RAW)
print(OUT_REPORT)

print("\nRecommended submit order:")
print("1) submission_b3_direct_qwen_cautious.csv")
print("2) submission_b3_direct_qwen_medium.csv")
print("3) submission_b3_direct_qwen_aggressive.csv")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n✅ Qwen Cell 2 complete. Runtime:", round(time.time() - t0, 2), "sec")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
device: cuda
gpu: Tesla T4
review pack: (400, 9)
base direct: (1500, 2)

Loading Qwen2.5-7B-Instruct 4-bit...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

...

Qwen loaded.


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


processed 25/400 | saved 25
processed 50/400 | saved 50
processed 75/400 | saved 75
processed 100/400 | saved 100
processed 125/400 | saved 125
processed 150/400 | saved 150
processed 175/400 | saved 175
processed 200/400 | saved 200
processed 225/400 | saved 225
processed 250/400 | saved 250
processed 275/400 | saved 275
processed 300/400 | saved 300
processed 325/400 | saved 325
processed 350/400 | saved 350
processed 375/400 | saved 375
processed 400/400 | saved 400

Qwen selection counts:


,count
select_mode,
qwen_exact,282
qwen_soft_match,81
qwen_label_prefix,24
invalid,13


changed_from_direct: 105
cautious rows: 1500 empty: 0
medium rows: 1500 empty: 0
aggressive rows: 1500 empty: 0

QWEN SELECTIVE REPORT
base : /content/drive/MyDrive/bangla_qa_070_sprint/sprint5_reader_ensemble_outputs/submission_b3_direct.csv
review_pack : /content/drive/MyDrive/bangla_qa_070_sprint/sprint5_reader_ensemble_outputs/qwen_selective_review_pack.parquet
model : Qwen/Qwen2.5-7B-Instruct
review_rows : 400
qwen_rows : 400
select_mode_counts : {'qwen_exact': 282, 'qwen_soft_match': 81, 'qwen_label_prefix': 24, 'invalid': 13}
changed_from_direct_raw : 105
submission_changes : {'cautious': 52, 'medium': 68, 'aggressive': 105}
outputs : {'cautious': '/content/drive/MyDrive/bangla_qa_070_sprint/sprint5_reader_ensemble_outputs/submission_b3_direct_qwen_cautious.csv', 'medium': '/content/drive/MyDrive/bangla_qa_070_sprint/sprint5_reader_ensemble_outputs/submission_b3_direct_qwen_medium.csv', 'aggressive': '/content/drive/MyDrive/bangla_qa_070_sprint/sprint5_reader_ensemble_outputs/su

In [ ]:
# ============================================================
# Download Qwen submission files. Qwen medium gave best results.
# ============================================================

from google.colab import files
from pathlib import Path
import os

SPRINT_B_DIR = Path("/content/drive/MyDrive/bangla_qa_070_sprint/sprint5_reader_ensemble_outputs")

download_files = [
    SPRINT_B_DIR / "submission_b3_direct_qwen_cautious.csv",
    SPRINT_B_DIR / "submission_b3_direct_qwen_medium.csv",
    SPRINT_B_DIR / "submission_b3_direct_qwen_aggressive.csv",
    SPRINT_B_DIR / "qwen_selective_report.json",
    SPRINT_B_DIR / "qwen_selective_raw_outputs.csv",
]

for f in download_files:
    if f.exists():
        print("Downloading:", f.name)
        files.download(str(f))
    else:
        print("Missing:", f)

Downloading: submission_b3_direct_qwen_cautious.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: submission_b3_direct_qwen_medium.csv


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: qwen_selective_raw_outputs.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>